In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/DrugRecommandation')
else:
    BASE_DIR = Path.cwd()

CKPT_DIR = BASE_DIR / 'checkpoints_phobert'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CSV_PATH = 'merged_drug_data_mapped(2).csv'

if not DATA_CSV_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy file dữ liệu: {DATA_CSV_PATH}')

print('IN_COLAB =', IN_COLAB)
print('BASE_DIR =', BASE_DIR)
print('CKPT_DIR =', CKPT_DIR)
print('DATA_CSV_PATH =', DATA_CSV_PATH)

Mounted at /content/drive
IN_COLAB = True
BASE_DIR = /content
CKPT_DIR = /content/checkpoints_phobert
DATA_CSV_PATH = /content/merged_drug_data_mapped(2).csv


In [4]:

K_EVAL = int(globals().get("K_EVAL", 5))


def _extract_query_and_label(item):
    if isinstance(item, dict):
        query = item.get("query") or item.get("query_text") or item.get("text") or item.get("prompt") or ""
        label = (
            item.get("expected_category")
            or item.get("label")
            or item.get("category")
            or item.get("target_category")
            or item.get("expected_label")
        )
        return str(query), None if label is None else str(label)

    if isinstance(item, (list, tuple)):
        if len(item) >= 2:
            return str(item[0]), None if item[1] is None else str(item[1])
        if len(item) == 1:
            return str(item[0]), None

    return str(item), None


def _get_category_column(df: pd.DataFrame) -> str:
    for col in ["expected_category", "category", "label", "pred_category"]:
        if col in df.columns:
            return col
    raise ValueError("Không tìm thấy cột category hợp lệ trong kết quả trả về của recommender")


def evaluate_labeled_queries_generic(labeled_queries, k, recommender, method_name: str):
    rows = []
    for item in labeled_queries:
        query, expected_cat = _extract_query_and_label(item)
        out = recommender(query, k=k)
        if out is None or len(out) == 0:
            rows.append({
                "query": query,
                "expected_category": expected_cat,
                "top1_category": None,
                "rank": None,
                "hit@1": 0.0,
                "hit@3": 0.0,
                f"hit@{k}": 0.0,
                f"mrr@{k}": 0.0,
                f"precision@{k}": 0.0,
                "top1_drug": None,
                "top1_score": np.nan,
            })
            continue

        cat_col = _get_category_column(out)
        pred_cats = out[cat_col].astype(str).head(k).tolist()
        rank = None
        if expected_cat is not None:
            for i, cat in enumerate(pred_cats, start=1):
                if cat == expected_cat:
                    rank = i
                    break

        rows.append({
            "query": query,
            "expected_category": expected_cat,
            "top1_category": pred_cats[0] if pred_cats else None,
            "rank": rank,
            "hit@1": 1.0 if rank == 1 else 0.0,
            "hit@3": 1.0 if (rank is not None and rank <= 3) else 0.0,
            f"hit@{k}": 1.0 if rank is not None else 0.0,
            f"mrr@{k}": (1.0 / rank) if rank is not None else 0.0,
            f"precision@{k}": float(sum(cat == expected_cat for cat in pred_cats)) / float(k) if expected_cat is not None else 0.0,
            "top1_drug": out.iloc[0].get("drug_name", None),
            "top1_score": float(out.iloc[0].get("score", np.nan)),
        })

    res = pd.DataFrame(rows)
    if len(res) > 0:
        print(
            f"[{method_name}] "
            f"hit@1={res['hit@1'].mean():.3f} | "
            f"hit@3={res['hit@3'].mean():.3f} | "
            f"hit@{k}={res[f'hit@{k}'].mean():.3f} | "
            f"MRR@{k}={res[f'mrr@{k}'].mean():.3f} | "
            f"Precision@{k}={res[f'precision@{k}'].mean():.3f}"
        )
    return res


In [5]:
data_df = pd.read_csv(DATA_CSV_PATH)
print(data_df.shape)
data_df.info()
data_df.head()

(21408, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21408 entries, 0 to 21407
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   drug_name          21408 non-null  object 
 1   active_ingredient  15017 non-null  object 
 2   dosage_form        15384 non-null  object 
 3   packing            11428 non-null  object 
 4   indication         20467 non-null  object 
 5   usage              9523 non-null   object 
 6   side_effect        16276 non-null  object 
 7   contraindication   16817 non-null  object 
 8   caution            16865 non-null  object 
 9   manufacturer       13160 non-null  object 
 10  price              2722 non-null   float64
 11  url                21408 non-null  object 
 12  image_url          11165 non-null  object 
 13  source             21408 non-null  object 
 14  category           21408 non-null  object 
dtypes: float64(1), object(14)
memory usage: 2.5+ MB


,drug_name,active_ingredient,dosage_form,packing,indication,usage,side_effect,contraindication,caution,manufacturer,price,url,image_url,source,category
0,"Buscopan 10mg trị co thắt đường tiêu hoá, sinh...",Hyoscin butylbromid,Viên nén bao đường,NaN,"Co thắt đường tiêu hóa, co thắt và rối loạn vậ...",Nên dùng theo liều khuyến cáo như sau trừ khi ...,Nhiều tác dụng ngoại ý được liệt kê dưới đây l...,BUSCOPAN chống chỉ định khi: - Bệnh nhân đã bi...,- Thận trọng khi sử dụng Trong các trường hợp ...,Sanofi,1500.0,https://www.nhathuocankhang.com/thuoc-tri-viem...,https://cdn.tgdd.vn/Products/Images/10039/1535...,ankhang,Tiêu hóa & gan mật
1,Spamerin 135mg trị các triệu chứng của hội chứ...,Mebeverin hydroclorid,Viên nén bao phim,NaN,Mebeverin được dùng trong điều trị các triệu c...,Người lớn và trẻ em trên 18 tuổi: 1 viên x 3 l...,"Rất hiếm: Rối loạn tiêu hóa, chóng mặt, nhức đ...",Quá mẫn cảm với mebeverin hydroclorid hay với ...,- Thận trọng khi sử dụng Tránh dùng thuốc ở bệ...,Abbott,23000.0,https://www.nhathuocankhang.com/thuoc-tri-benh...,https://cdn.tgdd.vn/Products/Images/10041/2433...,ankhang,Tiêu hóa & gan mật
2,"Silymax hỗ trợ trị bệnh viêm gan mãn tính, xơ ...",Silymarin,Viên nén bao đường,NaN,Hỗ trợ điều trị các bệnh về gan: Viêm gan do v...,- Người lớn: 2 viên/lần x 2 - 3 lần/ngày. - Qu...,Với liều thông thường rất hiếm gặp tác dụng kh...,Mẫn cảm với bất kỳ thành phần nào của thuốc. T...,- Thận trọng khi sử dụng Không dùng quá liều c...,Mediplantex,99000.0,https://www.nhathuocankhang.com/thuoc-tri-gan-...,https://cdn.tgdd.vn/Products/Images/10040/3123...,ankhang,Tiêu hóa & gan mật
3,Viên nhai Kremil-S làm dịu các triệu chứng đầy...,"Nhôm hydroxyd, Magnesi hydroxyd, Simethicon",Viên nén nhai,NaN,Làm dịu các triệu chứng tăng tiết acid dạ dày ...,Liều dùng cho người lớn: dùng 1 - 2 viên sau m...,"Buồn nôn, nôn, miệng có vị kim loại, tiêu chảy...",Mẫn cảm với bất cứ thành phần nào của thuốc. S...,- Thận trọng khi sử dụng Bệnh nhân có suy tim ...,United International Pharma,1200.0,https://www.nhathuocankhang.com/thuoc-tri-viem...,https://cdn.tgdd.vn/Products/Images/10039/1313...,ankhang,Tiêu hóa & gan mật
4,"Đại Tràng Nhất Nhất trị viêm đại tràng, rối lo...","Xa tiền tử, Ngũ bội tử, Cam thảo, Bạch truật, ...",Viên nén bao phim,NaN,"Trị viêm đại tràng, tiêu chảy, rối loạn tiêu h...",Nên uống vào lúc đói. Trẻ em 3-15 tuổi: Ngày u...,Chưa ghi nhận được báo cáo về phản ứng có hại ...,"Trẻ em dưới 30 tháng tuổi, trẻ em có tiền sử đ...",- Thận trọng khi sử dụng Thận trọng khi sử dụn...,Nhất Nhất,125000.0,https://www.nhathuocankhang.com/thuoc-tri-benh...,https://cdn.tgdd.vn/Products/Images/10041/2189...,ankhang,Tiêu hóa & gan mật


In [6]:
import re


def normalize_for_match(x: str):
    if pd.isna(x):
        return ""
    t = str(x).lower().strip()
    t = re.sub(r"\s+", " ", t)
    return t

dosage_patterns = [
    r"\b\d+([.,]\d+)?\s?(mg|mcg|g|kg|ml|l|iu|ui|%)\b",
    r"\b\d+([.,]\d+)?\s?/\s?\d+([.,]\d+)?\s?(mg|ml|g)\b",
]
form_patterns = [
    r"\b(viên nén|vien nen|viên nang|vien nang|sirô|siro|ống uống|ong uong|xịt|xit|kem|gel|bột|bot|gói|goi)\b",
]
pack_patterns = [
    r"\b(hộp|hop|chai|vỉ|vi|tuýp|tuyp)\s?\d*\b",
    r"\b\d+\s?(viên|vien|gói|goi|ống|ong|ml)\b",
]


def clean_name_for_dup(name):
    if pd.isna(name):
        return np.nan
    x = normalize_for_match(name)
    for p in dosage_patterns + form_patterns + pack_patterns:
        x = re.sub(p, " ", x)
    x = re.sub(r"[^0-9a-zà-ỹ\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else np.nan

train_cols = ["drug_name", "active_ingredient", "indication", "contraindication", "category", "source"]
missing = [c for c in train_cols if c not in data_df.columns]
if missing:
    raise ValueError(f"Thiếu cột bắt buộc trong data_df: {missing}")

train_df = data_df[train_cols].copy()
for col in ["drug_name", "active_ingredient", "indication", "contraindication", "category", "source"]:
    train_df[col] = train_df[col].fillna("").astype(str).str.strip()

# Bỏ record thiếu nhãn/tên và dedup
train_df = train_df[(train_df["drug_name"] != "") & (train_df["category"] != "")].copy()
train_df = train_df.drop_duplicates(subset=["drug_name", "active_ingredient", "indication", "contraindication", "category", "source"])

# Text feature lexical baseline (không thêm nhãn để tránh leakage)
train_df["text_features_model"] = (
    train_df["drug_name"] + " " +
    train_df["drug_name"] + " " +
    train_df["active_ingredient"] + " " +
    train_df["active_ingredient"] + " " +
    train_df["indication"] + " " +
    train_df["contraindication"]
).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

train_df["text_features"] = train_df["text_features_model"]
train_df["text_features_with_label"] = (train_df["text_features_model"] + " " + train_df["category"]).str.replace(r"\s+", " ", regex=True).str.strip()

# Key để group split giảm near-duplicate leakage
train_df["name_norm_clean"] = train_df["drug_name"].apply(clean_name_for_dup)
train_df["name_norm_clean"] = train_df["name_norm_clean"].fillna(train_df["drug_name"].str.lower().str.strip())

print("Prepared train_df:", train_df.shape)

# Kiểm tra xem clean_name_for_dup có đang gom quá nhiều thuốc khác nhau vào cùng 1 key không
collision = (
    train_df.groupby("name_norm_clean")["drug_name"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)
print("Top name_norm_clean collisions (unique drug_name per key):")
display(collision.to_frame(name="n_unique_drug_name"))

if len(collision) > 0:
    suspect_key = collision.index[0]
    print("\nSuspect key:", suspect_key)
    display(
        train_df.loc[train_df["name_norm_clean"] == suspect_key, ["drug_name", "category"]]
        .drop_duplicates()
        .sort_values(["category", "drug_name"])
    )

train_df.head()

Prepared train_df: (21408, 10)
Top name_norm_clean collisions (unique drug_name per key):


,n_unique_drug_name
name_norm_clean,
povidone agimexpharm,7
trajenta duo điều trị đái tháo đường type 2 viên,6
thuốc kháng sinh dmc cefalexin viên,5
wegovy novo nordisk 1 bút,5
thuốc trajenta duo boehringer điều trị đái tháo đường típ 2,5
thuốc hapacol dhg viên,5
thuốc hô hấp imexpharm dexipharm viên,5
pitorix pymepharco 3 x,4
thuốc viagra viatris điều trị rối loạn cương dương 1 x,4



Suspect key: povidone agimexpharm


,drug_name,category
16609,Povidone Agimexpharm 1100ml,Da liễu & dị ứng
16610,Povidone Agimexpharm 125ml,Da liễu & dị ứng
16611,Povidone Agimexpharm 130ml,Da liễu & dị ứng
16612,Povidone Agimexpharm 200ml,Da liễu & dị ứng
16613,Povidone Agimexpharm 20ml,Da liễu & dị ứng
16614,Povidone Agimexpharm 260ml,Da liễu & dị ứng
16587,Povidone Agimexpharm 90ml,Da liễu & dị ứng


,drug_name,active_ingredient,indication,contraindication,category,source,text_features_model,text_features,text_features_with_label,name_norm_clean
0,"Buscopan 10mg trị co thắt đường tiêu hoá, sinh...",Hyoscin butylbromid,"Co thắt đường tiêu hóa, co thắt và rối loạn vậ...",BUSCOPAN chống chỉ định khi: - Bệnh nhân đã bi...,Tiêu hóa & gan mật,ankhang,"buscopan 10mg trị co thắt đường tiêu hoá, sinh...","buscopan 10mg trị co thắt đường tiêu hoá, sinh...","buscopan 10mg trị co thắt đường tiêu hoá, sinh...",buscopan trị co thắt đường tiêu hoá sinh dục t...
1,Spamerin 135mg trị các triệu chứng của hội chứ...,Mebeverin hydroclorid,Mebeverin được dùng trong điều trị các triệu c...,Quá mẫn cảm với mebeverin hydroclorid hay với ...,Tiêu hóa & gan mật,ankhang,spamerin 135mg trị các triệu chứng của hội chứ...,spamerin 135mg trị các triệu chứng của hội chứ...,spamerin 135mg trị các triệu chứng của hội chứ...,spamerin trị các triệu chứng của hội chứng ruộ...
2,"Silymax hỗ trợ trị bệnh viêm gan mãn tính, xơ ...",Silymarin,Hỗ trợ điều trị các bệnh về gan: Viêm gan do v...,Mẫn cảm với bất kỳ thành phần nào của thuốc. T...,Tiêu hóa & gan mật,ankhang,"silymax hỗ trợ trị bệnh viêm gan mãn tính, xơ ...","silymax hỗ trợ trị bệnh viêm gan mãn tính, xơ ...","silymax hỗ trợ trị bệnh viêm gan mãn tính, xơ ...",silymax hỗ trợ trị bệnh viêm gan mãn tính xơ g...
3,Viên nhai Kremil-S làm dịu các triệu chứng đầy...,"Nhôm hydroxyd, Magnesi hydroxyd, Simethicon",Làm dịu các triệu chứng tăng tiết acid dạ dày ...,Mẫn cảm với bất cứ thành phần nào của thuốc. S...,Tiêu hóa & gan mật,ankhang,viên nhai kremil-s làm dịu các triệu chứng đầy...,viên nhai kremil-s làm dịu các triệu chứng đầy...,viên nhai kremil-s làm dịu các triệu chứng đầy...,viên nhai kremil s làm dịu các triệu chứng đầy...
4,"Đại Tràng Nhất Nhất trị viêm đại tràng, rối lo...","Xa tiền tử, Ngũ bội tử, Cam thảo, Bạch truật, ...","Trị viêm đại tràng, tiêu chảy, rối loạn tiêu h...","Trẻ em dưới 30 tháng tuổi, trẻ em có tiền sử đ...",Tiêu hóa & gan mật,ankhang,"đại tràng nhất nhất trị viêm đại tràng, rối lo...","đại tràng nhất nhất trị viêm đại tràng, rối lo...","đại tràng nhất nhất trị viêm đại tràng, rối lo...",đại tràng nhất nhất trị viêm đại tràng rối loạ...


In [9]:
# Chuẩn hoá tên cột nhãn duy nhất cho toàn notebook
if "category" not in data_df.columns:
    if "category_grouped_model" in data_df.columns:
        data_df = data_df.rename(columns={"category_grouped_model": "category"})
    elif "category_grouped" in data_df.columns:
        data_df = data_df.rename(columns={"category_grouped": "category"})

LABEL_COL = "category"
assert LABEL_COL in data_df.columns, "Không tìm thấy cột nhãn category"

print(data_df.shape)
print("Label column:", LABEL_COL)
print("Unique labels:", data_df[LABEL_COL].nunique())
data_df[[LABEL_COL]].head()

(21408, 15)
Label column: category
Unique labels: 16


,category
0,Tiêu hóa & gan mật
1,Tiêu hóa & gan mật
2,Tiêu hóa & gan mật
3,Tiêu hóa & gan mật
4,Tiêu hóa & gan mật


In [10]:
# Query-template specs + generators (đặt sớm trước bước build text/split/train)
BASE_PATTERNS = [
    "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
    "{duration} nay bị {symptom1}, kèm {symptom2} và {symptom3}.",
    "Triệu chứng gồm {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
    "Gần đây tôi bị {symptom1}, {symptom2}, {symptom3}; muốn tư vấn thuốc phù hợp.",
    "Bị {symptom1} và {symptom2}, thêm {symptom3}, tình trạng {modifier}.",
    "Tôi gặp {symptom1}, {symptom2}, {symptom3} trong {duration}, nghi {context}.",
]

COMMON_SYMPTOM_BANK = [
    ("đau đầu", "mệt mỏi", "khó ngủ", "stress"),
    ("sốt nhẹ", "đau người", "uể oải", "viêm nhiễm"),
    ("buồn nôn", "đầy bụng", "khó tiêu", "rối loạn tiêu hóa"),
    ("ho", "đau họng", "sổ mũi", "cảm lạnh"),
    ("ngứa", "nổi ban", "kích ứng da", "dị ứng"),
    ("đau cơ", "cứng cơ", "đau khi vận động", "quá sức"),
]

CATEGORY_QUERY_SPECS = {
    "Hô hấp": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("ho khan", "đau họng", "sổ mũi", "cúm mùa"),
            ("ho có đờm", "khò khè", "khó thở", "viêm phế quản"),
            ("ngạt mũi", "hắt hơi", "ngứa mũi", "viêm mũi dị ứng"),
            ("đau rát họng", "khàn tiếng", "ho nhẹ", "viêm họng"),
            ("sốt nhẹ", "mệt mỏi", "đau người", "cảm lạnh"),
            ("ho kéo dài", "tức ngực", "thở gấp", "nhiễm khuẩn hô hấp"),
        ],
    },
    "Tiêu hóa & gan mật": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("đau thượng vị", "ợ chua", "nóng rát dạ dày", "trào ngược"),
            ("buồn nôn", "đầy bụng", "khó tiêu", "rối loạn tiêu hóa"),
            ("tiêu chảy", "đau bụng", "mất nước nhẹ", "viêm dạ dày ruột"),
            ("táo bón", "chướng bụng", "đau bụng âm ỉ", "rối loạn tiêu hóa"),
            ("đau bụng", "nôn", "ăn kém", "viêm loét dạ dày"),
            ("đầy hơi", "ợ hơi", "khó chịu sau ăn", "khó tiêu"),
        ],
    },
    "Da liễu & dị ứng": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("ngứa da", "mề đay", "nổi ban đỏ", "dị ứng thức ăn"),
            ("viêm da tiếp xúc", "đỏ rát", "ngứa", "dị ứng"),
            ("mụn viêm", "da dầu", "sưng đỏ", "mụn trứng cá"),
            ("nổi mẩn", "ngứa toàn thân", "phát ban", "dị ứng thuốc"),
            ("chàm", "khô da", "ngứa", "viêm da cơ địa"),
            ("nấm da", "rát", "tróc vảy", "nhiễm nấm"),
        ],
    },
    "Cơ xương khớp": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("đau lưng", "co cứng cơ", "đau tăng khi vận động", "căng cơ"),
            ("đau khớp gối", "sưng nhẹ", "hạn chế vận động", "thoái hóa khớp"),
            ("đau vai gáy", "cứng cổ", "mỏi cơ", "thoái hóa cột sống"),
            ("đau cơ", "nhức mỏi", "mệt", "quá sức"),
            ("gút", "đau khớp", "sưng đỏ", "tăng acid uric"),
            ("đau cổ tay", "tê", "viêm gân", "chấn thương nhẹ"),
        ],
    },
    "Tim mạch & huyết áp": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("tăng huyết áp", "đau đầu", "chóng mặt", "huyết áp cao"),
            ("huyết áp dao động", "mệt", "khó chịu", "rối loạn huyết áp"),
            ("tim đập nhanh", "hồi hộp", "khó thở", "rối loạn nhịp"),
            ("đau thắt ngực", "tức ngực", "mệt", "thiếu máu cơ tim"),
            ("phù chân", "nặng ngực", "khó thở", "suy tim"),
            ("mỡ máu cao", "choáng", "nhức đầu", "rối loạn lipid"),
        ],
    },
    "Thần kinh & Não bộ": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("đau đầu", "mất ngủ", "căng thẳng", "stress kéo dài"),
            ("chóng mặt", "buồn nôn", "mệt", "rối loạn tiền đình"),
            ("lo âu", "bồn chồn", "khó ngủ", "căng thẳng thần kinh"),
            ("đau nửa đầu", "sợ ánh sáng", "buồn nôn", "migraine"),
            ("run tay", "mệt", "khó tập trung", "rối loạn thần kinh"),
            ("tê bì", "đau đầu", "nhức mỏi", "rối loạn thần kinh"),
        ],
    },
    "Tiết niệu & Sinh dục": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("tiểu buốt", "tiểu rắt", "đau bụng dưới", "nhiễm trùng tiểu"),
            ("tiểu nhiều lần", "khó chịu", "mệt", "rối loạn tiết niệu"),
            ("đau khi đi tiểu", "nước tiểu đục", "sốt nhẹ", "viêm đường tiết niệu"),
            ("tiểu đêm", "khát nước", "mệt", "tuyến tiền liệt"),
            ("đau vùng hạ vị", "tiểu khó", "đái rát", "viêm bàng quang"),
            ("ngứa rát", "khí hư", "khó chịu", "viêm nhiễm sinh dục"),
        ],
    },
    "Kháng sinh & Kháng viêm": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("sốt", "đau họng", "viêm nhiễm", "nhiễm khuẩn"),
            ("sưng đỏ", "đau", "nóng", "viêm mô mềm"),
            ("nhiễm trùng", "mệt", "đau", "viêm"),
            ("ho", "đờm", "viêm phế quản", "nhiễm khuẩn hô hấp"),
            ("viêm da", "đỏ", "ngứa", "nhiễm vi khuẩn"),
            ("đau răng", "sưng lợi", "viêm", "nhiễm khuẩn răng miệng"),
        ],
    },
    "Giảm đau & Hạ sốt": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("sốt", "đau đầu", "mệt", "cảm sốt"),
            ("đau nhức", "sốt nhẹ", "ớn lạnh", "cúm"),
            ("đau răng", "đau đầu", "sưng", "đau cấp"),
            ("đau bụng", "khó chịu", "mệt", "sốt nhẹ"),
            ("đau vai", "đau lưng", "nhức mỏi", "đau cơ"),
            ("đau đầu căng thẳng", "mệt", "mất ngủ", "stress"),
        ],
    },
    "Vitamin & Khoáng chất": {
        "patterns": BASE_PATTERNS,
        "symptoms": [
            ("mệt mỏi", "ăn kém", "thiếu vitamin", "suy nhược"),
            ("rụng tóc", "da khô", "móng yếu", "thiếu khoáng"),
            ("hoa mắt", "chóng mặt", "thiếu máu", "thiếu sắt"),
            ("mệt", "suy nhược", "kém ăn", "thiếu dinh dưỡng"),
            ("căng thẳng", "mất ngủ", "tâm trạng kém", "thiếu vitamin"),
            ("phục hồi sức khỏe", "yếu", "ăn uống kém", "bồi bổ"),
        ],
    },
}


def expand_category_specs(specs: dict, target_symptoms: int = 18) -> dict:
    expanded = {}
    for cat, spec in specs.items():
        patterns = list(spec.get("patterns", BASE_PATTERNS))
        symptoms = list(spec.get("symptoms", []))
        if len(symptoms) == 0:
            symptoms = list(COMMON_SYMPTOM_BANK)

        pool = list(symptoms)
        i = 0
        while len(pool) < int(target_symptoms):
            s1, s2, s3, ctx = symptoms[i % len(symptoms)]
            pool.append((s2, s3, s1, ctx))
            i += 1

        expanded[cat] = {
            "patterns": patterns,
            "symptoms": pool[: int(target_symptoms)],
        }
    return expanded


CATEGORY_QUERY_SPECS = expand_category_specs(CATEGORY_QUERY_SPECS, target_symptoms=18)
print("Expanded CATEGORY_QUERY_SPECS to >= 18 symptom tuples/category")
print(pd.Series({k: len(v["symptoms"]) for k, v in CATEGORY_QUERY_SPECS.items()}).sort_values(ascending=False).head())

Expanded CATEGORY_QUERY_SPECS to >= 18 symptom tuples/category
Hô hấp                 18
Tiêu hóa & gan mật     18
Da liễu & dị ứng       18
Cơ xương khớp          18
Tim mạch & huyết áp    18
dtype: int64


In [11]:
# Tạo rõ 2 loại text: document_text (thuốc) và query_text (triệu chứng tự nhiên)
def _safe_text_col(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index, dtype="object")


def build_document_text_trimmed(df: pd.DataFrame, max_ind: int = 200, max_contra: int = 100) -> pd.Series:
    name = _safe_text_col(df, "drug_name")
    ai = _safe_text_col(df, "active_ingredient")
    ind = _safe_text_col(df, "indication").str[: int(max_ind)]
    contra = _safe_text_col(df, "contraindication").str[: int(max_contra)]
    text = (
        "Tên thuốc: " + name + ". "
        + "Hoạt chất: " + ai + ". "
        + "Chỉ định: " + ind + ". "
        + "Chống chỉ định: " + contra
    )
    return text.str.replace(r"\s+", " ", regex=True).str.strip()


def build_query_symptom_only(df: pd.DataFrame) -> pd.Series:
    symptom_like = _safe_text_col(df, "indication").str.replace(r"\s+", " ", regex=True).str.strip()
    has_symptom = symptom_like.str.len() > 0
    query = np.where(
        has_symptom,
        "Tôi có triệu chứng: " + symptom_like,
        "Tôi cần tư vấn thuốc phù hợp",
    )
    return pd.Series(query, index=df.index, dtype="object").str.replace(r"\s+", " ", regex=True).str.strip()


def build_query_with_label(df: pd.DataFrame) -> pd.Series:
    symptom_like = _safe_text_col(df, "indication").str.replace(r"\s+", " ", regex=True).str.strip()
    category_like = _safe_text_col(df, "category").str.replace(r"\s+", " ", regex=True).str.strip()
    has_symptom = symptom_like.str.len() > 0
    has_category = category_like.str.len() > 0
    query = np.where(
        has_symptom & has_category,
        "Tôi có triệu chứng: " + symptom_like + ". Cần thuốc nhóm: " + category_like,
        np.where(
            has_symptom,
            "Tôi có triệu chứng: " + symptom_like,
            np.where(has_category, "Tôi cần thuốc cho nhóm bệnh: " + category_like, "Tôi cần tư vấn thuốc phù hợp"),
        ),
    )
    return pd.Series(query, index=df.index, dtype="object").str.replace(r"\s+", " ", regex=True).str.strip()


def assign_template_query(df: pd.DataFrame, specs: dict, seed: int = 42) -> pd.Series:
    rng = np.random.default_rng(seed)
    duration_opts = ["2 ngày", "3 ngày", "1 tuần", "vài hôm", "gần đây"]
    modifier_opts = ["kéo dài", "không đỡ", "tái phát", "nặng hơn về đêm"]
    context_opts = ["cảm lạnh", "dị ứng", "viêm nhiễm", "stress"]

    queries = []
    for _, row in df.iterrows():
        cat = str(row.get("category", "")).strip()
        spec = specs.get(cat)
        if spec is None:
            queries.append("Tôi cần tư vấn thuốc phù hợp")
            continue

        symptoms = spec.get("symptoms", [])
        patterns = spec.get("patterns", [])
        if len(symptoms) == 0 or len(patterns) == 0:
            queries.append("Tôi cần tư vấn thuốc phù hợp")
            continue

        s1, s2, s3, ctx = symptoms[int(rng.integers(0, len(symptoms)))]
        pattern = patterns[int(rng.integers(0, len(patterns)))]
        q = pattern.format(
            symptom1=s1,
            symptom2=s2,
            symptom3=s3,
            duration=str(rng.choice(duration_opts)),
            modifier=str(rng.choice(modifier_opts)),
            context=str(rng.choice([ctx] + context_opts)),
        )
        queries.append(str(q).strip())

    return pd.Series(queries, index=df.index, dtype="object").str.replace(r"\s+", " ", regex=True).str.strip()


# fallback text_features nếu file không có sẵn cột này
if "text_features" not in data_df.columns:
    data_df["text_features"] = (
        _safe_text_col(data_df, "drug_name") + " " +
        _safe_text_col(data_df, "active_ingredient") + " " +
        _safe_text_col(data_df, "indication") + " " +
        _safe_text_col(data_df, "contraindication")
    ).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

# Query columns
if "CATEGORY_QUERY_SPECS" not in globals():
    CATEGORY_QUERY_SPECS = {}

data_df["query_symptom_only"] = build_query_symptom_only(data_df)
data_df["query_with_label"] = build_query_with_label(data_df)
data_df["query_from_template"] = assign_template_query(data_df, CATEGORY_QUERY_SPECS, seed=42)

# query_text giữ tương thích cũ, nhưng mô hình sẽ dùng query_from_template
data_df["query_text"] = data_df["query_symptom_only"]

# Document text (trimmed)
data_df["document_text"] = build_document_text_trimmed(data_df, max_ind=200, max_contra=100)

# Tạo key để group split ổn định hơn về sau
if "name_norm_clean" not in data_df.columns:
    data_df["name_norm_clean"] = data_df["drug_name"].apply(clean_name_for_dup)
    data_df["name_norm_clean"] = data_df["name_norm_clean"].fillna(
        data_df["drug_name"].fillna("").astype(str).str.lower().str.strip()
    )

print("Created/validated columns: text_features, document_text, query_symptom_only, query_with_label, query_from_template, query_text, name_norm_clean")
print(data_df[["document_text", "query_from_template", "query_symptom_only", "query_with_label"]].head(2))

Created/validated columns: text_features, document_text, query_symptom_only, query_with_label, query_from_template, query_text, name_norm_clean
                                       document_text  \
0  Tên thuốc: Buscopan 10mg trị co thắt đường tiê...   
1  Tên thuốc: Spamerin 135mg trị các triệu chứng ...   

                                 query_from_template  \
0  Bị buồn nôn và đầy bụng, thêm khó tiêu, tình t...   
1      chướng bụng, đau bụng âm ỉ, táo bón, kéo dài.   

                                  query_symptom_only  \
0  Tôi có triệu chứng: Co thắt đường tiêu hóa, co...   
1  Tôi có triệu chứng: Mebeverin được dùng trong ...   

                                    query_with_label  
0  Tôi có triệu chứng: Co thắt đường tiêu hóa, co...  
1  Tôi có triệu chứng: Mebeverin được dùng trong ...  


In [12]:
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
class PhoBERTFineTuner(nn.Module):
    def __init__(self, model_name="vinai/phobert-base", hidden_dim=768, use_attention=True, dropout=0.2):
        super(PhoBERTFineTuner, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.use_attention = bool(use_attention)
        if self.use_attention:
            self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)

        # Regularized projection head
        self.projection = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(float(dropout)),
            nn.Linear(hidden_dim, 768),
        )

    @staticmethod
    def _masked_mean_pool(token_embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).type_as(token_embeddings)
        masked = token_embeddings * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-6)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        backbone_frozen = not any(p.requires_grad for p in self.phobert.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state.detach()
        else:
            outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
            hidden_states = outputs.last_hidden_state

        if self.use_attention:
            key_padding_mask = (attention_mask == 0)
            attention_output, _ = self.attention(
                hidden_states, hidden_states, hidden_states,
                key_padding_mask=key_padding_mask,
                need_weights=False,
            )
            pooled = self._masked_mean_pool(attention_output, attention_mask)
        else:
            pooled = self._masked_mean_pool(hidden_states, attention_mask)

        embeddings = self.projection(pooled)
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings

In [14]:
model_attn = PhoBERTFineTuner()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_attn = model_attn.to(device)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
CPU_FAST_MODE = not torch.cuda.is_available()

# Bật fine-tune backbone để học semantic tốt hơn TF-IDF lexical-only
FREEZE_BACKBONE = False

# Hard negatives: cùng category, khác drug
USE_HARD_NEGATIVES = True
HARD_NEGATIVE_MARGIN = 0.4
HARD_NEGATIVE_WEIGHT = 0.15
HARD_NEGATIVE_WEIGHT_LATE = 0.2
HARD_NEGATIVE_WARMUP_EPOCHS = 2
HARD_NEGATIVE_STRATEGY = "same_category"

# Giảm nhẹ chi phí khi chạy CPU
MAX_LENGTH = 96 if CPU_FAST_MODE else 128
EPOCHS = 3 if CPU_FAST_MODE else 8
PATIENCE = 3 if not CPU_FAST_MODE else 2
MIN_DELTA = 1e-4
PAIRS_PER_ROW = 1
GRAD_ACCUM_STEPS = 4 if CPU_FAST_MODE else 1
max_length = MAX_LENGTH

# Đồng bộ text cho cả baseline và PhoBERT theo cùng bài toán query-doc
DOCUMENT_TEXT_COL = "document_text"
QUERY_TEXT_COL = "query_from_template"
PHOBERT_DOC_COL = DOCUMENT_TEXT_COL
PHOBERT_QUERY_COL = QUERY_TEXT_COL
TFIDF_DOC_COL = DOCUMENT_TEXT_COL
TFIDF_QUERY_COL = QUERY_TEXT_COL

# Learning rates (discriminative LR)
BACKBONE_LR = 1e-5 if CPU_FAST_MODE else 2e-5
HEAD_LR = 5e-5 if CPU_FAST_MODE else 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

print("DOCUMENT_TEXT_COL:", DOCUMENT_TEXT_COL)
print("QUERY_TEXT_COL:", QUERY_TEXT_COL)
print("CPU_FAST_MODE:", CPU_FAST_MODE)
print("FREEZE_BACKBONE:", FREEZE_BACKBONE)
print("USE_HARD_NEGATIVES:", USE_HARD_NEGATIVES)
print("HARD_NEGATIVE_STRATEGY:", HARD_NEGATIVE_STRATEGY)
print("MAX_LENGTH:", MAX_LENGTH, "| EPOCHS:", EPOCHS, "| GRAD_ACCUM_STEPS:", GRAD_ACCUM_STEPS)
print("BACKBONE_LR:", BACKBONE_LR, "| HEAD_LR:", HEAD_LR)
print("HARD_NEGATIVE_MARGIN:", HARD_NEGATIVE_MARGIN, "| HARD_NEGATIVE_WEIGHT:", HARD_NEGATIVE_WEIGHT)
print("PATIENCE:", PATIENCE, "| MIN_DELTA:", MIN_DELTA)

# Protocol so sánh công bằng giữa các mô hình (single source of truth)
BENCHMARK_PROTOCOL = {
    "K_EVAL": 5,
    "TFIDF_NGRAM_RANGE": (1, 2),
    "TFIDF_MIN_DF": 3,
    "TFIDF_MAX_FEATURES": 50000,
    "TFIDF_SVD_COMPONENTS": 300,
    "DL_BENCHMARK_EPOCHS": 12 if not CPU_FAST_MODE else 6,
    "DL_BENCHMARK_BATCH_SIZE": 64 if torch.cuda.is_available() else 16,
    "DL_TEXT_MAX_LENGTH": 64 if not CPU_FAST_MODE else 48,
    "DL_MIN_WORD_FREQ": 2,
    "DL_MAX_VOCAB_SIZE": 30000,
    "DL_EMBED_DIM": 128,
    "DL_HIDDEN_DIM": 128,
    "DL_CNN_CHANNELS": 128,
    "DL_PROJ_DIM": 256,
    "DL_LR": 1e-3,
    "DL_WEIGHT_DECAY": 0.01,
    "DL_TEMPERATURE": 0.08,
    "DL_DROPOUT": 0.25,
}
K_EVAL = BENCHMARK_PROTOCOL["K_EVAL"]

print("\n=== Benchmark protocol ===")
print(BENCHMARK_PROTOCOL)

DOCUMENT_TEXT_COL: document_text
QUERY_TEXT_COL: query_from_template
CPU_FAST_MODE: False
FREEZE_BACKBONE: False
USE_HARD_NEGATIVES: True
HARD_NEGATIVE_STRATEGY: same_category
MAX_LENGTH: 128 | EPOCHS: 8 | GRAD_ACCUM_STEPS: 1
BACKBONE_LR: 2e-05 | HEAD_LR: 0.0001
HARD_NEGATIVE_MARGIN: 0.4 | HARD_NEGATIVE_WEIGHT: 0.15
PATIENCE: 3 | MIN_DELTA: 0.0001

=== Benchmark protocol ===
{'K_EVAL': 5, 'TFIDF_NGRAM_RANGE': (1, 2), 'TFIDF_MIN_DF': 3, 'TFIDF_MAX_FEATURES': 50000, 'TFIDF_SVD_COMPONENTS': 300, 'DL_BENCHMARK_EPOCHS': 12, 'DL_BENCHMARK_BATCH_SIZE': 64, 'DL_TEXT_MAX_LENGTH': 64, 'DL_MIN_WORD_FREQ': 2, 'DL_MAX_VOCAB_SIZE': 30000, 'DL_EMBED_DIM': 128, 'DL_HIDDEN_DIM': 128, 'DL_CNN_CHANNELS': 128, 'DL_PROJ_DIM': 256, 'DL_LR': 0.001, 'DL_WEIGHT_DECAY': 0.01, 'DL_TEMPERATURE': 0.08, 'DL_DROPOUT': 0.25}


In [16]:
from sklearn.model_selection import GroupShuffleSplit


group_col = "name_norm_clean" if "name_norm_clean" in data_df.columns else "drug_name"
groups = data_df[group_col].fillna("").astype(str).values

# Bước 1: tách test 10%
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(gss_test.split(data_df, groups=groups))
train_val_df = data_df.iloc[train_val_idx].reset_index(drop=True)
test_set = data_df.iloc[test_idx].reset_index(drop=True)

# Bước 2: tách val 5% từ toàn bộ dữ liệu
groups_tv = train_val_df[group_col].fillna("").astype(str).values
gss_val = GroupShuffleSplit(n_splits=1, test_size=(5 / 90), random_state=42)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=groups_tv))

train_set = train_val_df.iloc[train_idx].reset_index(drop=True)
val_set = train_val_df.iloc[val_idx].reset_index(drop=True)

# Rebuild document/query cho từng split theo checklist
for df_ in [train_set, val_set, test_set]:
    df_["document_text"] = build_document_text_trimmed(df_, max_ind=200, max_contra=100)
    df_["query_from_template"] = assign_template_query(df_, CATEGORY_QUERY_SPECS, seed=42)

# Bổ sung/chuẩn hoá cột text và nhãn
for name, df_ in [("train", train_set), ("val", val_set), ("test", test_set)]:
    missing = [c for c in [LABEL_COL, DOCUMENT_TEXT_COL, QUERY_TEXT_COL] if c not in df_.columns]
    if missing:
        raise ValueError(f"{name}_set thiếu cột bắt buộc: {missing}")
    df_[DOCUMENT_TEXT_COL] = df_[DOCUMENT_TEXT_COL].fillna("").astype(str)
    df_[QUERY_TEXT_COL] = df_[QUERY_TEXT_COL].fillna("").astype(str)
    df_[LABEL_COL] = df_[LABEL_COL].fillna("Unknown").astype(str)

print(f"Split ratio target: train=85% | val=5% | test=10%")
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
print("Label distribution (train, top 10):")
print(train_set[LABEL_COL].value_counts().head(10))

# Bắt buộc kiểm tra overlap query-document trước khi train
sample = train_set.sample(min(200, len(train_set)), random_state=42)
overlaps = []
for q, d in zip(sample[PHOBERT_QUERY_COL], sample[DOCUMENT_TEXT_COL]):
    q_w = set(str(q).lower().split())
    d_w = set(str(d).lower().split())
    if len(q_w) > 0:
        overlaps.append(len(q_w & d_w) / len(q_w))
if len(overlaps) > 0:
    print(f"Overlap train sample: mean={np.mean(overlaps):.1%}, p50={np.quantile(overlaps, 0.5):.1%}, p90={np.quantile(overlaps, 0.9):.1%}")

# Kiểm tra độ dài document sau trim
doc_lens = train_set[DOCUMENT_TEXT_COL].fillna("").str.len()
print(f"doc_len p50={doc_lens.quantile(0.5):.0f} | p90={doc_lens.quantile(0.9):.0f}")

Split ratio target: train=85% | val=5% | test=10%
Train: 18210 | Val: 1067 | Test: 2131
Label distribution (train, top 10):
category
Kháng sinh & Kháng viêm    2457
Tim mạch & huyết áp        2150
Da liễu & dị ứng           2123
Tiêu hóa & gan mật         2078
Thần kinh & Não bộ         1361
Giảm đau & Hạ sốt          1330
Vitamin & Khoáng chất      1149
Hô hấp                     1110
Cơ xương khớp               894
Khác                        893
Name: count, dtype: int64
Overlap train sample: mean=11.1%, p50=9.1%, p90=27.4%
doc_len p50=389 | p90=468


In [17]:
import torch.nn.functional as F
from collections import defaultdict


class QueryDocumentPairDataset(Dataset):
    def __init__(
        self,
        data: pd.DataFrame,
        query_col: str,
        doc_col: str,
        label_col: str,
        pairs_per_row: int = 1,
        seed: int = 42,
        use_hard_negatives: bool = False,
        hard_negative_strategy: str = "same_category",
    ):
        keep_cols = [query_col, doc_col, label_col]
        self.data = data.dropna(subset=keep_cols).reset_index(drop=True)
        self.query_col = query_col
        self.doc_col = doc_col
        self.label_col = label_col
        self.pairs_per_row = int(max(1, pairs_per_row))
        self.seed = int(seed)
        self.use_hard_negatives = bool(use_hard_negatives)
        self.hard_negative_strategy = str(hard_negative_strategy)
        self.rng = np.random.default_rng(self.seed)

        self.category_col = "category" if "category" in self.data.columns else None
        self.drug_name_col = "drug_name" if "drug_name" in self.data.columns else None
        self.category_to_indices = {}
        if self.use_hard_negatives and self.category_col is not None:
            grouped_indices = defaultdict(list)
            for idx, category in enumerate(self.data[self.category_col].astype(str).tolist()):
                grouped_indices[category].append(idx)
            self.category_to_indices = {
                category: np.asarray(indices, dtype=np.int64)
                for category, indices in grouped_indices.items()
            }

        self.pairs = self._create_pairs()

    def _sample_hard_negative(self, idx: int) -> str:
        if not self.use_hard_negatives or len(self.data) <= 1:
            return ""

        if self.hard_negative_strategy == "same_category" and self.category_col is not None:
            row = self.data.iloc[idx]
            category = str(row[self.category_col])
            candidate_indices = self.category_to_indices.get(category, np.asarray([], dtype=np.int64))
            if len(candidate_indices) > 0:
                if self.drug_name_col is not None and self.drug_name_col in self.data.columns:
                    candidate_mask = self.data.iloc[candidate_indices][self.drug_name_col].astype(str).values != str(row[self.drug_name_col])
                    candidate_indices = candidate_indices[candidate_mask]
                if len(candidate_indices) > 0:
                    neg_idx = int(self.rng.choice(candidate_indices))
                    return str(self.data.iloc[neg_idx][self.doc_col]).strip()

        candidates = self.data.index[self.data.index != idx].to_numpy()
        if len(candidates) == 0:
            return ""
        neg_idx = int(self.rng.choice(candidates))
        return str(self.data.iloc[neg_idx][self.doc_col]).strip()

    def _create_pairs(self):
        # Positive pair đúng nghĩa: query_text và document_text của cùng record
        pairs = []
        for i in range(len(self.data)):
            q = str(self.data.iloc[i][self.query_col]).strip()
            d = str(self.data.iloc[i][self.doc_col]).strip()
            if not q or not d:
                continue
            neg = self._sample_hard_negative(i)
            for _ in range(self.pairs_per_row):
                if self.use_hard_negatives:
                    pairs.append((q, d, neg, str(self.data.iloc[i][self.label_col])))
                else:
                    pairs.append((q, d, str(self.data.iloc[i][self.label_col])))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]
        if self.use_hard_negatives:
            q, d, neg, y = item
            return {"query_text": q, "doc_text": d, "neg_text": neg, "label": y}
        q, d, y = item
        return {"query_text": q, "doc_text": d, "label": y}


def multiple_negatives_ranking_loss(
    query_emb,
    doc_emb,
    temperature: float = 0.05,
    symmetric: bool = True,
):
    logits = (query_emb @ doc_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    loss = F.cross_entropy(logits, labels)
    if symmetric:
        loss = 0.5 * (loss + F.cross_entropy(logits.T, labels))
    return loss


def hard_negative_margin_loss(query_emb, pos_emb, neg_emb, margin: float = 0.4):
    pos_sim = (query_emb * pos_emb).sum(dim=1)
    neg_sim = (query_emb * neg_emb).sum(dim=1)
    return torch.relu(float(margin) - pos_sim + neg_sim).mean()

In [18]:
from transformers import get_linear_schedule_with_warmup

if FREEZE_BACKBONE:
    for p in model_attn.phobert.parameters():
        p.requires_grad = False
    model_attn.phobert.eval()
else:
    for p in model_attn.phobert.parameters():
        p.requires_grad = True

pair_dataset = QueryDocumentPairDataset(
    train_set,
    query_col=PHOBERT_QUERY_COL,
    doc_col=PHOBERT_DOC_COL,
    label_col=LABEL_COL,
    pairs_per_row=PAIRS_PER_ROW,
    seed=42,
    use_hard_negatives=USE_HARD_NEGATIVES,
    hard_negative_strategy=HARD_NEGATIVE_STRATEGY,
)

val_pair_dataset = QueryDocumentPairDataset(
    val_set,
    query_col=PHOBERT_QUERY_COL,
    doc_col=PHOBERT_DOC_COL,
    label_col=LABEL_COL,
    pairs_per_row=1,
    seed=42,
    use_hard_negatives=False,
)

batch_size = 64 if torch.cuda.is_available() else 8
dataloader = DataLoader(
    pair_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)
val_dataloader = DataLoader(
    val_pair_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    pin_memory=torch.cuda.is_available(),
)

backbone_params = [p for n, p in model_attn.named_parameters() if n.startswith("phobert") and p.requires_grad]
head_params = [p for n, p in model_attn.named_parameters() if (not n.startswith("phobert")) and p.requires_grad]
param_groups = []
if len(backbone_params) > 0:
    param_groups.append({"params": backbone_params, "lr": float(BACKBONE_LR)})
if len(head_params) > 0:
    param_groups.append({"params": head_params, "lr": float(HEAD_LR)})

optimizer = torch.optim.AdamW(param_groups, weight_decay=float(WEIGHT_DECAY))

# fix(2): tăng temperature để giảm mode collapse embedding
temperature = 0.07
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

epochs = int(EPOCHS)
num_update_steps_per_epoch = max(1, int(np.ceil(len(dataloader) / int(GRAD_ACCUM_STEPS))))
max_train_steps = int(epochs * num_update_steps_per_epoch)
warmup_steps = int(max(0, WARMUP_RATIO) * max_train_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=max_train_steps,
)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

model_attn.train()
if FREEZE_BACKBONE:
    model_attn.phobert.eval()
print(f"Pair dataset size: {len(pair_dataset)} | Val pairs: {len(val_pair_dataset)}")
print(f"query_col={PHOBERT_QUERY_COL} | doc_col={PHOBERT_DOC_COL}")
print(f"CPU_FAST_MODE={CPU_FAST_MODE} | FREEZE_BACKBONE={FREEZE_BACKBONE} | batch={batch_size}")
print(f"EPOCHS={epochs} | grad_accum={GRAD_ACCUM_STEPS} | steps={max_train_steps} | warmup={warmup_steps}")
print(f"backbone_params={len(backbone_params)} | head_params={len(head_params)}")
print(f"PhoBERT temperature={temperature}")

Pair dataset size: 18210 | Val pairs: 1067
query_col=query_from_template | doc_col=document_text
CPU_FAST_MODE=False | FREEZE_BACKBONE=False | batch=64
EPOCHS=8 | grad_accum=1 | steps=2272 | warmup=136
backbone_params=199 | head_params=8
PhoBERT temperature=0.07


/tmp/ipykernel_4296/4119087319.py:61: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


In [19]:
# Sanity-check query-doc pairs
print("num_train_pairs:", len(pair_dataset))

sample_pairs = pair_dataset.pairs[:5]
if len(sample_pairs) > 0 and len(sample_pairs[0]) == 4:
    sample_df = pd.DataFrame(sample_pairs, columns=["query_text", "doc_text", "neg_text", "label"])
else:
    sample_df = pd.DataFrame(sample_pairs, columns=["query_text", "doc_text", "label"])

print("Sample pairs:")
display(sample_df)

print("Unique labels in training pairs:", pd.Series([p[-1] for p in pair_dataset.pairs]).nunique())

num_train_pairs: 18210
Sample pairs:


,query_text,doc_text,neg_text,label
0,"Bị buồn nôn và đầy bụng, thêm khó tiêu, tình t...",Tên thuốc: Buscopan 10mg trị co thắt đường tiê...,Tên thuốc: Shinphagel Shinpoong Daewoo 20 gói ...,Tiêu hóa & gan mật
1,"chướng bụng, đau bụng âm ỉ, táo bón, kéo dài.",Tên thuốc: Spamerin 135mg trị các triệu chứng ...,Tên thuốc: Esomeprazol 20-US trị trào ngược dạ...,Tiêu hóa & gan mật
2,"Tôi gặp chướng bụng, đau bụng âm ỉ, táo bón tr...",Tên thuốc: Silymax hỗ trợ trị bệnh viêm gan mã...,Tên thuốc: Thuốc Slandom 8 Savi phòng buồn nôn...,Tiêu hóa & gan mật
3,"Gần đây tôi bị đau bụng, mất nước nhẹ, tiêu ch...",Tên thuốc: Viên nhai Kremil-S làm dịu các triệ...,Tên thuốc: Viên nén Nexium Mups 40mg AstraZene...,Tiêu hóa & gan mật
4,"Triệu chứng gồm chướng bụng, đau bụng âm ỉ, tá...",Tên thuốc: Đại Tràng Nhất Nhất trị viêm đại tr...,Tên thuốc: Hỗn dịch uống Sucrapi trị viêm loét...,Tiêu hóa & gan mật


Unique labels in training pairs: 16


In [ ]:
from time import perf_counter
from pathlib import Path

def evaluate_pair_loss(model, loader, tokenizer, device, max_length, temperature=0.05):
    model.eval()
    total = 0.0
    n_steps = 0
    with torch.no_grad():
        for batch in loader:
            query_texts = batch["query_text"]
            doc_texts = batch["doc_text"]
            texts = list(query_texts) + list(doc_texts)
            toks = tokenizer(
                texts,
                max_length=max_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            toks = {k: v.to(device) for k, v in toks.items()}
            bsz = len(query_texts)
            embs = model(toks["input_ids"], toks["attention_mask"])
            query_emb = embs[:bsz]
            doc_emb = embs[bsz:]
            loss = multiple_negatives_ranking_loss(query_emb, doc_emb, temperature=temperature, symmetric=True)
            total += float(loss.item())
            n_steps += 1
    model.train()
    if FREEZE_BACKBONE:
        model.phobert.eval()
    return total / max(1, n_steps)


if "hard_negative_margin_loss" not in globals():
    def hard_negative_margin_loss(query_emb, pos_emb, neg_emb, margin: float = 0.4):
        pos_sim = (query_emb * pos_emb).sum(dim=1)
        neg_sim = (query_emb * neg_emb).sum(dim=1)
        return torch.relu(float(margin) - pos_sim + neg_sim).mean()


epochs = int(EPOCHS)
max_length = int(MAX_LENGTH)
accum_steps = int(max(1, GRAD_ACCUM_STEPS))

CKPT_OUT_DIR = Path(globals().get("CKPT_DIR", Path.cwd() / "checkpoints_phobert"))
CKPT_OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CKPT_OUT_DIR / "phobert_finetuned(2)_best.pth"
LATEST_CKPT_PATH = CKPT_OUT_DIR / "phobert_finetuned(2)_latest.pth"

best_val_loss = float("inf")
no_improve_epochs = 0

for epoch in range(epochs):
    total_loss = 0.0
    t0 = perf_counter()

    if FREEZE_BACKBONE:
        model_attn.phobert.eval()

    optimizer.zero_grad(set_to_none=True)

    current_hard_negative_weight = HARD_NEGATIVE_WEIGHT if epoch < HARD_NEGATIVE_WARMUP_EPOCHS else HARD_NEGATIVE_WEIGHT_LATE

    for step, batch in enumerate(dataloader, start=1):
        query_texts = batch["query_text"]
        doc_texts = batch["doc_text"]
        neg_texts = batch.get("neg_text", None)
        texts = list(query_texts) + list(doc_texts)
        if neg_texts is not None:
            texts += list(neg_texts)

        toks = tokenizer(
            texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        toks = {k: v.to(device) for k, v in toks.items()}

        bsz = len(query_texts)

        if use_amp:
            with torch.cuda.amp.autocast(enabled=True):
                embs = model_attn(toks["input_ids"], toks["attention_mask"])
                query_emb = embs[:bsz]
                doc_emb = embs[bsz:2 * bsz]
                loss = multiple_negatives_ranking_loss(
                    query_emb,
                    doc_emb,
                    temperature=temperature,
                    symmetric=True,
                )
                if neg_texts is not None:
                    neg_emb = embs[2 * bsz:3 * bsz]
                    loss = loss + current_hard_negative_weight * hard_negative_margin_loss(
                        query_emb,
                        doc_emb,
                        neg_emb,
                        margin=HARD_NEGATIVE_MARGIN,
                    )
                loss = loss / accum_steps
            scaler.scale(loss).backward()
        else:
            embs = model_attn(toks["input_ids"], toks["attention_mask"])
            query_emb = embs[:bsz]
            doc_emb = embs[bsz:2 * bsz]
            loss = multiple_negatives_ranking_loss(
                query_emb,
                doc_emb,
                temperature=temperature,
                symmetric=True,
            )
            if neg_texts is not None:
                neg_emb = embs[2 * bsz:3 * bsz]
                loss = loss + current_hard_negative_weight * hard_negative_margin_loss(
                    query_emb,
                    doc_emb,
                    neg_emb,
                    margin=HARD_NEGATIVE_MARGIN,
                )
            loss = loss / accum_steps
            loss.backward()

        total_loss += float(loss.item())

        do_step = (step % accum_steps == 0) or (step == len(dataloader))
        if do_step:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model_attn.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model_attn.parameters(), 1.0)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if step % 200 == 0:
            elapsed = perf_counter() - t0
            avg_loss = total_loss / step
            lr_now = optimizer.param_groups[0]["lr"]
            print(
                f"Epoch {epoch+1}/{epochs} - step {step}/{len(dataloader)} - "
                f"train_loss {avg_loss:.4f} - lr {lr_now:.2e} - {elapsed:.1f}s"
            )

    train_loss = total_loss / max(1, len(dataloader))
    val_loss = evaluate_pair_loss(
        model_attn, val_dataloader, tokenizer, device, max_length=max_length, temperature=temperature
    )
    epoch_time = perf_counter() - t0

    torch.save(model_attn.state_dict(), str(LATEST_CKPT_PATH))

    improved = val_loss < (best_val_loss - MIN_DELTA)
    if improved:
        best_val_loss = val_loss
        no_improve_epochs = 0
        torch.save(model_attn.state_dict(), str(BEST_CKPT_PATH))
    else:
        no_improve_epochs += 1

    print(
        f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"best_val={best_val_loss:.4f} | time={epoch_time/60:.2f} min | "
        f"hard_neg_weight={current_hard_negative_weight:.2f} | no_improve={no_improve_epochs}"
    )
    print(f"Saved latest: {LATEST_CKPT_PATH}")
    if improved:
        print(f"Saved best: {BEST_CKPT_PATH}")

    if no_improve_epochs >= PATIENCE:
        print(f"Early stopping triggered after {no_improve_epochs} epochs without improvement.")
        break

/tmp/ipykernel_4296/1390073964.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


Epoch 1/8 - step 200/284 - train_loss 3.3832 - lr 1.94e-05 - 216.9s
Epoch 1/8 | train_loss=3.1840 | val_loss=2.6387 | best_val=2.6387 | time=5.46 min | hard_neg_weight=0.15 | no_improve=0
Saved latest: /content/checkpoints_phobert/phobert_finetuned(2)_latest.pth
Saved best: /content/checkpoints_phobert/phobert_finetuned(2)_best.pth
Epoch 2/8 - step 200/284 - train_loss 2.5822 - lr 1.67e-05 - 223.9s
Epoch 2/8 | train_loss=2.5637 | val_loss=2.5072 | best_val=2.5072 | time=5.58 min | hard_neg_weight=0.15 | no_improve=0
Saved latest: /content/checkpoints_phobert/phobert_finetuned(2)_latest.pth
Saved best: /content/checkpoints_phobert/phobert_finetuned(2)_best.pth
Epoch 3/8 - step 200/284 - train_loss 2.4714 - lr 1.41e-05 - 224.7s
Epoch 3/8 | train_loss=2.4595 | val_loss=2.4560 | best_val=2.4560 | time=5.59 min | hard_neg_weight=0.20 | no_improve=0
Saved latest: /content/checkpoints_phobert/phobert_finetuned(2)_latest.pth
Saved best: /content/checkpoints_phobert/phobert_finetuned(2)_best.pt

In [21]:
def ranking_metrics_from_labels(pred_labels, true_label, k=5):
    topk = pred_labels[:k]
    hits = [1 if lbl == true_label else 0 for lbl in topk]

    hit_at_k = 1.0 if any(hits) else 0.0

    rank = None
    for i, lbl in enumerate(topk, start=1):
        if lbl == true_label:
            rank = i
            break
    mrr_at_k = 1.0 / rank if rank is not None else 0.0

    precision_at_k = float(sum(hits)) / float(k)

    return {
        "hit@k": hit_at_k,
        "mrr@k": mrr_at_k,
        "precision@k": precision_at_k,
    }

In [22]:
def encode_texts(model, tokenizer, texts, device, batch_size: int = 32, max_length: int = 128):
    model.eval()
    all_embs = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start : start + batch_size]
            toks = tokenizer(
                batch_texts,
                max_length=max_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            )
            toks = {k: v.to(device) for k, v in toks.items()}
            embs = (
                model(toks["input_ids"], toks["attention_mask"])
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )
            all_embs.append(embs)
    return np.vstack(all_embs)


PRECOMPUTE_TRAIN_EMB = False
if PRECOMPUTE_TRAIN_EMB:
    train_texts = train_set[DOCUMENT_TEXT_COL].fillna("").tolist()
    enc_bs = 64 if torch.cuda.is_available() else 16
    train_emb = encode_texts(
        model_attn,
        tokenizer,
        train_texts,
        device,
        batch_size=enc_bs,
        max_length=max_length,
    )
    print(f"Train embeddings: {train_emb.shape}, dtype={train_emb.dtype}")

In [23]:
MODEL_SAVE_PATH = CKPT_DIR / "phobert_finetuned(2)_latest.pth"
torch.save(model_attn.state_dict(), str(MODEL_SAVE_PATH))
print(f"Model saved successfully: {MODEL_SAVE_PATH}")

Model saved successfully: /content/checkpoints_phobert/phobert_finetuned(2)_latest.pth


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

vectorizer = TfidfVectorizer(
    ngram_range=BENCHMARK_PROTOCOL["TFIDF_NGRAM_RANGE"],
    min_df=BENCHMARK_PROTOCOL["TFIDF_MIN_DF"],
    max_features=BENCHMARK_PROTOCOL["TFIDF_MAX_FEATURES"],
    dtype=np.float32,
)

# Fit trên document index (train docs)
X_train_tfidf = vectorizer.fit_transform(train_set[TFIDF_DOC_COL].fillna(""))
X_val_query_tfidf = vectorizer.transform(val_set[TFIDF_QUERY_COL].fillna(""))
X_test_query_tfidf = vectorizer.transform(test_set[TFIDF_QUERY_COL].fillna(""))

print("TF-IDF train docs:", X_train_tfidf.shape, "nnz=", X_train_tfidf.nnz, "dtype=", X_train_tfidf.dtype)
print("TF-IDF val queries:", X_val_query_tfidf.shape)
print("TF-IDF test queries:", X_test_query_tfidf.shape)

svd = TruncatedSVD(n_components=BENCHMARK_PROTOCOL["TFIDF_SVD_COMPONENTS"], random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf).astype(np.float32)
X_val_query_svd = svd.transform(X_val_query_tfidf).astype(np.float32)
X_test_query_svd = svd.transform(X_test_query_tfidf).astype(np.float32)

X_train_svd = normalize(X_train_svd)
X_val_query_svd = normalize(X_val_query_svd)
X_test_query_svd = normalize(X_test_query_svd)

print("SVD train docs:", X_train_svd.shape, "dtype=", X_train_svd.dtype)
print("SVD val queries:", X_val_query_svd.shape)
print("SVD test queries:", X_test_query_svd.shape)

TF-IDF train docs: (18210, 48237) nnz= 1867875 dtype= float32
TF-IDF val queries: (1067, 48237)
TF-IDF test queries: (2131, 48237)
SVD train docs: (18210, 300) dtype= float32
SVD val queries: (1067, 300)
SVD test queries: (2131, 300)


In [25]:
# Baseline query -> document retrieval bằng TF-IDF(SVD)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

TFIDF_QUERY_CANDIDATES = 50
knn_tfidf_query = NearestNeighbors(
    n_neighbors=TFIDF_QUERY_CANDIDATES, metric="cosine", algorithm="brute"
)
knn_tfidf_query.fit(X_train_svd)
print("TF-IDF query KNN ready. candidates=", TFIDF_QUERY_CANDIDATES)


def recommend_from_query_tfidf(query: str, k: int = 5):
    qtext = query.strip()
    q_tfidf = vectorizer.transform([qtext])
    q_svd = svd.transform(q_tfidf).astype(np.float32)
    q_svd = normalize(q_svd)
    dist, idx = knn_tfidf_query.kneighbors(q_svd, n_neighbors=k)
    sims = 1.0 - dist[0]
    results = train_set.iloc[idx[0]].copy()
    results["score"] = sims
    return results.sort_values("score", ascending=False)

TF-IDF query KNN ready. candidates= 50


In [26]:
from sklearn.neighbors import NearestNeighbors

# --- Ensure doc embeddings for index ---
eval_cfg = {
    "encoder_version": int(globals().get("ENCODER_VERSION", 2)),
    "text_col": PHOBERT_DOC_COL,
    "max_length": int(max_length),
    "train_len": int(len(train_set)),
}

need_eval_encode = True
if "train_emb" in globals() and isinstance(train_emb, np.ndarray) and train_emb.shape[0] == len(train_set):
    if "train_emb_cfg" in globals() and isinstance(train_emb_cfg, dict):
        if train_emb_cfg.get("text_col") == PHOBERT_DOC_COL and int(train_emb_cfg.get("max_length", -1)) == int(max_length):
            need_eval_encode = False

if need_eval_encode:
    train_docs = train_set[PHOBERT_DOC_COL].fillna("").tolist()
    enc_bs = 64 if torch.cuda.is_available() else 16
    train_emb = encode_texts(
        model_attn, tokenizer, train_docs, device, batch_size=enc_bs, max_length=max_length
    )
    print("Encoded train_emb for eval:", train_emb.shape)


def evaluate_query_doc_retrieval(
    query_texts,
    true_labels,
    retrieve_fn,
    k=5,
):
    rows = []
    for q, y in zip(query_texts, true_labels):
        out = retrieve_fn(q, k)
        pred_labels = out[LABEL_COL].astype(str).tolist()
        m = ranking_metrics_from_labels(pred_labels, str(y), k=k)
        rows.append(m)

    res = pd.DataFrame(rows)
    hit1 = []
    hit3 = []
    hitk = []
    mrrk = []
    precisionk = []
    for _, row in res.iterrows():
        hitk.append(float(row["hit@k"]))
        mrrk.append(float(row["mrr@k"]))
        precisionk.append(float(row["precision@k"]))
        hit1.append(1.0 if row["mrr@k"] == 1.0 else 0.0)
        hit3.append(1.0 if row["mrr@k"] > 0.0 and row["mrr@k"] >= 1.0 / 3.0 else 0.0)

    return {
        "hit@1": float(np.mean(hit1)) if len(hit1) else np.nan,
        "hit@3": float(np.mean(hit3)) if len(hit3) else np.nan,
        f"hit@{k}": float(np.mean(hitk)) if len(hitk) else np.nan,
        f"mrr@{k}": float(np.mean(mrrk)) if len(mrrk) else np.nan,
        f"precision@{k}": float(np.mean(precisionk)) if len(precisionk) else np.nan,
    }


# PhoBERT query encoder + doc index
k = 5
knn_pho = NearestNeighbors(n_neighbors=k, metric="cosine", algorithm="brute")
knn_pho.fit(train_emb)

def recommend_from_query_phobert_eval(query: str, k: int = 5):
    q_emb = encode_texts(model_attn, tokenizer, [query], device, batch_size=1, max_length=max_length)
    dist, idx = knn_pho.kneighbors(q_emb, n_neighbors=k)
    sims = 1.0 - dist[0]
    results = train_set.iloc[idx[0]].copy()
    results["score"] = sims
    return results.sort_values("score", ascending=False)

val_queries = val_set[PHOBERT_QUERY_COL].fillna("").astype(str).tolist()
val_labels = val_set[LABEL_COL].fillna("Unknown").astype(str).tolist()

pho_metrics = evaluate_query_doc_retrieval(
    val_queries,
    val_labels,
    retrieve_fn=recommend_from_query_phobert_eval,
    k=k,
)

tfidf_metrics = evaluate_query_doc_retrieval(
    val_queries,
    val_labels,
    retrieve_fn=recommend_from_query_tfidf,
    k=k,
)

print("Validation metrics (query -> document, category-proxy):")
print("TF-IDF:", tfidf_metrics)
print("PhoBERT:", pho_metrics)


Encoded train_emb for eval: (18210, 768)
Validation metrics (query -> document, category-proxy):
TF-IDF: {'hit@1': 0.46672914714151825, 'hit@3': 0.5923149015932521, 'hit@5': 0.6232427366447985, 'mrr@5': 0.5333645735707591, 'precision@5': 0.4931583880037489}
PhoBERT: {'hit@1': 0.8603561387066542, 'hit@3': 0.9053420805998126, 'hit@5': 0.9306466729147141, 'mrr@5': 0.8877694470477976, 'precision@5': 0.8552952202436739}


In [27]:
from sklearn.neighbors import NearestNeighbors
import os, json, hashlib

# Ưu tiên checkpoint mới nhất đã lưu
WEIGHTS_PATH = str(CKPT_DIR / "phobert_finetuned(2)_latest.pth")
if not os.path.exists(WEIGHTS_PATH):
    fallback = "phobert_finetuned(14_3).pth"
    if os.path.exists(fallback):
        WEIGHTS_PATH = fallback

INDEX_TEXT_COL = PHOBERT_DOC_COL
INDEX_BATCH_SIZE = 64 if torch.cuda.is_available() else 16
INDEX_MAX_LENGTH = max_length
INDEX_K = 10

# Bump khi đổi encoder / text formulation
ENCODER_VERSION = 4

EMB_CACHE_PATH = str(CKPT_DIR / "train_emb_phobert14_3.npy")
CFG_CACHE_PATH = str(CKPT_DIR / "train_emb_phobert14_3.json")

if not os.path.exists(WEIGHTS_PATH):
    raise FileNotFoundError(f"Không tìm thấy weights để load: {WEIGHTS_PATH}")

model_attn.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model_attn.eval()

weights_mtime = os.path.getmtime(WEIGHTS_PATH) if os.path.exists(WEIGHTS_PATH) else None
weights_size = os.path.getsize(WEIGHTS_PATH) if os.path.exists(WEIGHTS_PATH) else None

fp_cols = [c for c in ["name_norm_clean", LABEL_COL, INDEX_TEXT_COL] if c in train_set.columns]
fp_payload = (
    train_set[fp_cols]
    .fillna("")
    .astype(str)
    .head(2000)
    .to_csv(index=False)
).encode("utf-8", errors="ignore")
train_fingerprint = hashlib.md5(fp_payload).hexdigest()

train_emb_cfg = {
    "encoder_version": int(ENCODER_VERSION),
    "weights_path": WEIGHTS_PATH,
    "weights_mtime": weights_mtime,
    "weights_size": weights_size,
    "text_col": INDEX_TEXT_COL,
    "batch_size": int(INDEX_BATCH_SIZE),
    "max_length": int(INDEX_MAX_LENGTH),
    "train_len": int(len(train_set)),
    "train_fingerprint": train_fingerprint,
}

def _try_load_cached_emb():
    if not (os.path.exists(EMB_CACHE_PATH) and os.path.exists(CFG_CACHE_PATH)):
        return None
    try:
        with open(CFG_CACHE_PATH, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if cfg != train_emb_cfg:
            return None
        emb = np.load(EMB_CACHE_PATH)
        if not isinstance(emb, np.ndarray) or emb.shape[0] != len(train_set):
            return None
        return emb
    except Exception as e:
        print("Cache load failed -> will re-encode:", type(e).__name__, e)
        return None

need_encode = True
if "train_emb" in globals() and isinstance(train_emb, np.ndarray) and train_emb.shape[0] == len(train_set):
    if "train_emb_cfg" in globals() and globals()["train_emb_cfg"] == train_emb_cfg:
        need_encode = False
    else:
        print("train_emb exists but cfg mismatch -> re-encode")
        need_encode = True
else:
    cached = _try_load_cached_emb()
    if cached is not None:
        train_emb = cached
        need_encode = False
        print("Loaded cached train_emb:", train_emb.shape)

if need_encode:
    texts = train_set[INDEX_TEXT_COL].fillna("").tolist()
    train_emb = encode_texts(
        model_attn,
        tokenizer,
        texts,
        device,
        batch_size=INDEX_BATCH_SIZE,
        max_length=INDEX_MAX_LENGTH,
    )
    globals()["train_emb_cfg"] = train_emb_cfg
    print("Encoded train_emb:", train_emb.shape)
    try:
        np.save(EMB_CACHE_PATH, train_emb)
        with open(CFG_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(train_emb_cfg, f, ensure_ascii=False, indent=2)
        print("Saved embedding cache:", EMB_CACHE_PATH)
    except Exception as e:
        print("Cache save failed (ok to ignore):", type(e).__name__, e)
else:
    globals()["train_emb_cfg"] = train_emb_cfg
    print("Reusing train_emb (validated cfg):", train_emb.shape)

knn = NearestNeighbors(n_neighbors=INDEX_K, metric="cosine", algorithm="brute")
knn.fit(train_emb)
print("KNN index ready for document retrieval. k=", INDEX_K)

Reusing train_emb (validated cfg): (18210, 768)
KNN index ready for document retrieval. k= 10


In [28]:
# Diagnostics: check embedding similarity spread (helps explain weird high scores ~0.98)
if "train_emb" not in globals() or not isinstance(train_emb, np.ndarray) or train_emb.shape[0] == 0:
    print("Skip diagnostics: train_emb chưa sẵn sàng.")
elif "knn" not in globals():
    print("Skip diagnostics: knn chưa được khởi tạo.")
else:
    rng = np.random.default_rng(42)
    n = int(train_emb.shape[0])
    pair_samples = min(5000, n * 2)
    i1 = rng.integers(0, n, size=pair_samples)
    i2 = rng.integers(0, n, size=pair_samples)
    pair_sims = (train_emb[i1] * train_emb[i2]).sum(axis=1)  # cosine sim because embeddings are normalized
    print("Random-pair cosine sim stats:")
    print("  mean=", float(np.mean(pair_sims)), "std=", float(np.std(pair_sims)))
    print("  p50=", float(np.quantile(pair_sims, 0.50)), "p90=", float(np.quantile(pair_sims, 0.90)), "p99=", float(np.quantile(pair_sims, 0.99)))

    # Nearest-neighbor distance stats (excluding self-match)
    probe = min(500, n)
    probe_idx = rng.choice(n, size=probe, replace=False)
    dist2, idx2 = knn.kneighbors(train_emb[probe_idx], n_neighbors=2)
    nn_dist = dist2[:, 1]  # nearest neighbor other than itself
    print("\nNearest-neighbor cosine distance (k=2) stats:")
    print("  mean=", float(np.mean(nn_dist)), "p50=", float(np.quantile(nn_dist, 0.50)), "p90=", float(np.quantile(nn_dist, 0.90)), "p99=", float(np.quantile(nn_dist, 0.99)))
    print("  (cosine similarity = 1 - distance)")

Random-pair cosine sim stats:
  mean= 0.3304116129875183 std= 0.2512069642543793
  p50= 0.27307313680648804 p90= 0.7740232944488525 p99= 0.9689763188362122

Nearest-neighbor cosine distance (k=2) stats:
  mean= 0.012185884639620781 p50= 0.007491946220397949 p90= 0.026983080431818962 p99= 0.07277660071849823
  (cosine similarity = 1 - distance)


In [29]:
def recommend_from_query(
    query,
    knn,
    train_emb,
    train_df,
    model,
    tokenizer,
    device,
    k=5,
    max_length=None,
    batch_size=1,

    query_prefix="",
    index_max_length=None,
 ):
    qtext = (query_prefix or "") + query.strip()

    if max_length is None:
        if index_max_length is not None:
            max_length = int(index_max_length)
        else:
            max_length = int(globals().get("INDEX_MAX_LENGTH", 128))

    q_emb = encode_texts(
        model, tokenizer, [qtext], device, batch_size=batch_size, max_length=max_length
    )
    dist, idx = knn.kneighbors(q_emb, n_neighbors=k)
    sims = 1.0 - dist[0]

    results = train_df.iloc[idx[0]].copy()
    results["score"] = sims
    return results.sort_values("score", ascending=False)

In [31]:
# Evaluate symptom-query retrieval (query -> document)
# Sinh khoảng 200-300 query để đánh giá ổn định hơn thay vì chỉ vài chục mẫu thủ công.

CATEGORY_QUERY_SPECS = {
    "Hô hấp": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "Bị {symptom1}, kèm {symptom2} và {symptom3} kéo dài.",
            "{symptom1} {duration}, {symptom2}, {symptom3}, {modifier}.",
            "Triệu chứng gồm {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("ho khan", "đau họng", "sổ mũi", "cúm mùa"),
            ("ho có đờm", "khò khè", "khó thở", "viêm phế quản"),
            ("ngạt mũi", "hắt hơi", "ngứa mũi", "viêm mũi dị ứng"),
            ("đau rát họng", "khàn tiếng", "ho nhẹ", "viêm họng"),
            ("sốt nhẹ", "mệt mỏi", "đau người", "cảm lạnh"),
            ("ho kéo dài", "tức ngực", "thở gấp", "nhiễm khuẩn hô hấp"),
        ],
    },
    "Tiêu hóa & gan mật": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "Sau khi ăn bị {symptom1}, {symptom2}, {symptom3}.",
            "{duration} có {symptom1}, {symptom2}, {symptom3}.",
            "Triệu chứng tiêu hóa: {symptom1}, {symptom2}, {symptom3}.",
        ],
        "symptoms": [
            ("đau thượng vị", "ợ chua", "nóng rát dạ dày", "trào ngược"),
            ("buồn nôn", "đầy bụng", "khó tiêu", "rối loạn tiêu hóa"),
            ("tiêu chảy", "đau bụng", "mất nước nhẹ", "viêm dạ dày ruột"),
            ("táo bón", "chướng bụng", "đau bụng âm ỉ", "rối loạn tiêu hóa"),
            ("đau bụng", "nôn", "ăn kém", "viêm loét dạ dày"),
            ("đầy hơi", "ợ hơi", "khó chịu sau ăn", "khó tiêu"),
        ],
    },
    "Da liễu & dị ứng": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "Bị {symptom1} sau khi {context}, kèm {symptom2}.",
            "{duration} xuất hiện {symptom1}, {symptom2}, {symptom3}.",
            "Triệu chứng da liễu: {symptom1}, {symptom2}, {symptom3}.",
        ],
        "symptoms": [
            ("ngứa da", "mề đay", "nổi ban đỏ", "dị ứng thức ăn"),
            ("viêm da tiếp xúc", "đỏ rát", "ngứa", "dị ứng"),
            ("mụn viêm", "da dầu", "sưng đỏ", "mụn trứng cá"),
            ("nổi mẩn", "ngứa toàn thân", "phát ban", "dị ứng thuốc"),
            ("chàm", "khô da", "ngứa", "viêm da cơ địa"),
            ("nấm da", "rát", "tróc vảy", "nhiễm nấm"),
        ],
    },
    "Cơ xương khớp": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "Sau khi vận động bị {symptom1}, kèm {symptom2}.",
            "{duration} có {symptom1}, {symptom2}, {symptom3}.",
            "Triệu chứng cơ xương khớp: {symptom1}, {symptom2}, {symptom3}.",
        ],
        "symptoms": [
            ("đau lưng", "co cứng cơ", "đau tăng khi vận động", "căng cơ"),
            ("đau khớp gối", "sưng nhẹ", "hạn chế vận động", "thoái hóa khớp"),
            ("đau vai gáy", "cứng cổ", "mỏi cơ", "thoái hóa cột sống"),
            ("đau cơ", "nhức mỏi", "mệt", "quá sức"),
            ("gút", "đau khớp", "sưng đỏ", "tăng acid uric"),
            ("đau cổ tay", "tê", "viêm gân", "chấn thương nhẹ"),
        ],
    },
    "Tim mạch & huyết áp": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "{duration} có {symptom1}, kèm {symptom2} và {symptom3}.",
            "Triệu chứng tim mạch: {symptom1}, {symptom2}, {symptom3}.",
            "Bị {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("tăng huyết áp", "đau đầu", "chóng mặt", "huyết áp cao"),
            ("huyết áp dao động", "mệt", "khó chịu", "rối loạn huyết áp"),
            ("tim đập nhanh", "hồi hộp", "khó thở", "rối loạn nhịp"),
            ("đau thắt ngực", "tức ngực", "mệt", "thiếu máu cơ tim"),
            ("phù chân", "nặng ngực", "khó thở", "suy tim"),
            ("mỡ máu cao", "choáng", "nhức đầu", "rối loạn lipid"),
        ],
    },
    "Thần kinh & Não bộ": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "{duration} bị {symptom1}, kèm {symptom2} và {symptom3}.",
            "Triệu chứng thần kinh: {symptom1}, {symptom2}, {symptom3}.",
            "Bị {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("đau đầu", "mất ngủ", "căng thẳng", "stress kéo dài"),
            ("chóng mặt", "buồn nôn", "mệt", "rối loạn tiền đình"),
            ("lo âu", "bồn chồn", "khó ngủ", "căng thẳng thần kinh"),
            ("đau nửa đầu", "sợ ánh sáng", "buồn nôn", "migraine"),
            ("run tay", "mệt", "khó tập trung", "rối loạn thần kinh"),
            ("tê bì", "đau đầu", "nhức mỏi", "rối loạn thần kinh"),
        ],
    },
    "Tiết niệu & Sinh dục": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "{duration} có {symptom1}, kèm {symptom2}.",
            "Triệu chứng tiết niệu: {symptom1}, {symptom2}, {symptom3}.",
            "Bị {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("tiểu buốt", "tiểu rắt", "đau bụng dưới", "nhiễm trùng tiểu"),
            ("tiểu nhiều lần", "khó chịu", "mệt", "rối loạn tiết niệu"),
            ("đau khi đi tiểu", "nước tiểu đục", "sốt nhẹ", "viêm đường tiết niệu"),
            ("tiểu đêm", "khát nước", "mệt", "tuyến tiền liệt"),
            ("đau vùng hạ vị", "tiểu khó", "đái rát", "viêm bàng quang"),
            ("ngứa rát", "khí hư", "khó chịu", "viêm nhiễm sinh dục"),
        ],
    },
    "Kháng sinh & Kháng viêm": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "{duration} có {symptom1}, {symptom2} và {symptom3}.",
            "Triệu chứng viêm nhiễm: {symptom1}, {symptom2}, {symptom3}.",
            "Bị {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("sốt", "đau họng", "viêm nhiễm", "nhiễm khuẩn"),
            ("sưng đỏ", "đau", "nóng", "viêm mô mềm"),
            ("nhiễm trùng", "mệt", "đau", "viêm"),
            ("ho", "đờm", "viêm phế quản", "nhiễm khuẩn hô hấp"),
            ("viêm da", "đỏ", "ngứa", "nhiễm vi khuẩn"),
            ("đau răng", "sưng lợi", "viêm", "nhiễm khuẩn răng miệng"),
        ],
    },
    "Giảm đau & Hạ sốt": {
        "patterns": [
            "{symptom1}, {symptom2}, {symptom3}, {modifier}.",
            "{duration} bị {symptom1}, kèm {symptom2}.",
            "Triệu chứng đau sốt: {symptom1}, {symptom2}, {symptom3}.",
            "Bị {symptom1}, {symptom2}, {symptom3}, nghi {context}.",
        ],
        "symptoms": [
            ("sốt", "đau đầu", "mệt", "cảm sốt"),
            ("đau nhức", "sốt nhẹ", "ớn lạnh", "cúm"),
            ("đau răng", "đau đầu", "sưng", "đau cấp"),
            ("đau bụng", "khó chịu", "mệt", "sốt nhẹ"),
            ("đau vai", "đau lưng", "nhức mỏi", "đau cơ"),
            ("đau đầu căng thẳng", "mệt", "mất ngủ", "stress"),
        ],
    },
    "Vitamin & Khoáng chất": {
        "patterns": [
            "{context}, cần bổ sung {symptom1}, {symptom2}, {symptom3}.",
            "{duration} thấy {symptom1}, {symptom2}, {symptom3}.",
            "Triệu chứng thiếu chất: {symptom1}, {symptom2}, {symptom3}.",
            "Bổ sung cho {context}, biểu hiện {symptom1}, {symptom2}.",
        ],
        "symptoms": [
            ("mệt mỏi", "ăn kém", "thiếu vitamin", "suy nhược"),
            ("rụng tóc", "da khô", "móng yếu", "thiếu khoáng"),
            ("hoa mắt", "chóng mặt", "thiếu máu", "thiếu sắt"),
            ("mệt", "suy nhược", "kém ăn", "thiếu dinh dưỡng"),
            ("căng thẳng", "mất ngủ", "tâm trạng kém", "thiếu vitamin"),
            ("phục hồi sức khỏe", "yếu", "ăn uống kém", "bồi bổ"),
        ],
    },
}


def build_labeled_queries(target_count: int = 240, seed: int = 42):
    rng = np.random.default_rng(seed)
    items = []
    category_names = list(CATEGORY_QUERY_SPECS.keys())

    duration_options = ["2 ngày", "3 ngày", "1 tuần", "vài hôm", "nhiều ngày", "hôm nay", "gần đây"]
    modifier_options = ["kéo dài", "nặng hơn về đêm", "kèm sốt nhẹ", "không đỡ", "tái phát", "ảnh hưởng sinh hoạt"]
    context_options = ["cảm lạnh", "dị ứng", "viêm nhiễm", "rối loạn tiêu hóa", "stress", "làm việc quá sức"]
    extra_phrases = [
        "mong được tư vấn thuốc phù hợp",
        "nên dùng thuốc gì",
        "có thuốc nào cải thiện nhanh",
        "muốn tìm thuốc không kê đơn",
        "cần gợi ý thuốc theo triệu chứng",
    ]

    per_category = int(np.ceil(target_count / len(category_names)))
    for category in category_names:
        spec = CATEGORY_QUERY_SPECS[category]
        for i in range(per_category):
            pattern = spec["patterns"][i % len(spec["patterns"])]
            symptom1, symptom2, symptom3, base_context = spec["symptoms"][i % len(spec["symptoms"])]
            query = pattern.format(
                symptom1=symptom1,
                symptom2=symptom2,
                symptom3=symptom3,
                duration=str(rng.choice(duration_options)),
                modifier=str(rng.choice(modifier_options)),
                context=str(rng.choice([base_context] + context_options)),
            )
            query = f"{query[:-1]} — {rng.choice(extra_phrases)}." if query.endswith(".") else f"{query} — {rng.choice(extra_phrases)}."
            items.append((query, category))

    rng.shuffle(items)
    return items[:target_count]


LABELED_QUERIES = build_labeled_queries(target_count=240, seed=42)
print(f"Generated labeled queries: {len(LABELED_QUERIES)}")
print("Sample labeled queries:")
display(pd.DataFrame(LABELED_QUERIES[:10], columns=["query", "expected_category"]))


Generated labeled queries: 240
Sample labeled queries:


,query,expected_category
0,"hôm nay có đau lưng, co cứng cơ, đau tăng khi ...",Cơ xương khớp
1,"Bị đau thắt ngực, tức ngực, mệt, nghi dị ứng —...",Tim mạch & huyết áp
2,"Triệu chứng thiếu chất: mệt mỏi, ăn kém, thiếu...",Vitamin & Khoáng chất
3,"3 ngày có tiểu đêm, kèm khát nước — nên dùng t...",Tiết niệu & Sinh dục
4,"Triệu chứng cơ xương khớp: đau khớp gối, sưng ...",Cơ xương khớp
5,"Bị ngứa rát, khí hư, khó chịu, nghi cảm lạnh —...",Tiết niệu & Sinh dục
6,"Triệu chứng tiết niệu: đau khi đi tiểu, nước t...",Tiết niệu & Sinh dục
7,"Triệu chứng da liễu: nổi mẩn, ngứa toàn thân, ...",Da liễu & dị ứng
8,"Triệu chứng cơ xương khớp: đau cổ tay, tê, viê...",Cơ xương khớp
9,"viêm nhiễm, cần bổ sung căng thẳng, mất ngủ, t...",Vitamin & Khoáng chất


In [32]:
# fix(2): Tách bộ eval specs khỏi training specs (không share tuple/pattern)

def build_labeled_queries_from_specs(specs: dict, target_count: int = 240, seed: int = 42):
    rng = np.random.default_rng(seed)
    items = []
    category_names = list(specs.keys())

    duration_options = ["hôm nay", "2 hôm", "3 hôm", "1 tuần", "gần đây", "mấy ngày nay"]
    modifier_options = ["khó chịu nhiều", "ảnh hưởng sinh hoạt", "không đỡ", "hay tái lại", "nặng về chiều", "rõ hơn ban đêm"]
    context_options = ["viêm cấp", "dị ứng", "rối loạn chức năng", "nhiễm khuẩn", "căng thẳng", "thời tiết thay đổi"]
    extra_phrases = [
        "muốn được gợi ý thuốc phù hợp",
        "cần tư vấn loại thuốc không kê đơn",
        "mong cải thiện triệu chứng sớm",
        "xin gợi ý nhóm thuốc phù hợp",
        "nhờ tư vấn hướng dùng thuốc",
    ]

    per_category = int(np.ceil(target_count / max(1, len(category_names))))
    for category in category_names:
        spec = specs[category]
        patterns = spec.get("patterns", [])
        symptoms = spec.get("symptoms", [])
        if len(patterns) == 0 or len(symptoms) == 0:
            continue

        for i in range(per_category):
            pattern = patterns[i % len(patterns)]
            s1, s2, s3, base_context = symptoms[i % len(symptoms)]
            query = pattern.format(
                symptom1=s1,
                symptom2=s2,
                symptom3=s3,
                duration=str(rng.choice(duration_options)),
                modifier=str(rng.choice(modifier_options)),
                context=str(rng.choice([base_context] + context_options)),
            )
            if query.endswith("."):
                query = f"{query[:-1]} — {rng.choice(extra_phrases)}."
            else:
                query = f"{query} — {rng.choice(extra_phrases)}."
            items.append((query, category))

    rng.shuffle(items)
    return items[:target_count]


EVAL_BASE_PATTERNS = [
    "Tôi đang bị {symptom1}, đồng thời {symptom2} và {symptom3}; tình trạng {modifier}.",
    "Khoảng {duration} nay xuất hiện {symptom1}, kèm {symptom2}, {symptom3}.",
    "Biểu hiện gần đây: {symptom1}, {symptom2}, {symptom3}; nghi {context}.",
    "Xin tư vấn vì có {symptom1}, thêm {symptom2} và {symptom3}.",
]

EVAL_QUERY_SPECS = {
    "Hô hấp": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("viêm họng", "khan tiếng", "nuốt đau", "nhiễm khuẩn hầu họng"),
            ("thở khò khè", "tức ngực", "đờm xanh", "viêm phế quản"),
            ("nghẹt mũi", "rát mũi", "chảy dịch mũi", "viêm mũi xoang"),
            ("ho thành cơn", "rát họng", "khó thở nhẹ", "kích ứng đường thở"),
        ],
    },
    "Tiêu hóa & gan mật": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("đầy hơi", "ợ nóng", "đau vùng thượng vị", "trào ngược dạ dày"),
            ("đi ngoài phân lỏng", "quặn bụng", "mệt sau ăn", "viêm ruột"),
            ("khó tiêu", "ăn mau no", "buồn nôn", "rối loạn dạ dày"),
            ("đau âm ỉ bụng", "ợ chua", "cồn cào", "kích ứng tiêu hóa"),
        ],
    },
    "Da liễu & dị ứng": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("mẩn đỏ", "ngứa râm ran", "nóng rát da", "dị ứng tiếp xúc"),
            ("phát ban", "khô nứt da", "tróc vảy", "viêm da"),
            ("ngứa quanh cổ", "nổi sẩn", "kích ứng da", "dị ứng"),
            ("mụn đỏ", "đau rát", "tiết dầu nhiều", "mụn viêm"),
        ],
    },
    "Cơ xương khớp": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("mỏi vai", "cứng cổ", "đau khi xoay", "căng cơ cổ vai"),
            ("đau thắt lưng", "co cơ", "hạn chế cúi", "đau cơ xương"),
            ("nhức khớp gối", "sưng nhẹ", "đi lại khó", "viêm khớp"),
            ("đau vùng cổ tay", "tê nhẹ", "mỏi kéo dài", "quá tải gân cơ"),
        ],
    },
    "Tim mạch & huyết áp": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("choáng váng", "nhức đầu", "mặt nóng", "dao động huyết áp"),
            ("hồi hộp", "khó thở nhẹ", "mệt nhanh", "rối loạn nhịp"),
            ("nặng ngực", "mệt", "đau đầu âm ỉ", "tuần hoàn kém"),
            ("huyết áp cao", "mất tập trung", "khó chịu", "tăng huyết áp"),
        ],
    },
    "Thần kinh & Não bộ": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("đau đầu lệch bên", "sợ tiếng ồn", "buồn nôn", "migraine"),
            ("bồn chồn", "khó ngủ", "mệt sáng hôm sau", "rối loạn giấc ngủ"),
            ("hoa mắt", "mất thăng bằng", "uể oải", "tiền đình"),
            ("run nhẹ tay", "khó tập trung", "mỏi mệt", "căng thẳng thần kinh"),
        ],
    },
    "Tiết niệu & Sinh dục": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("tiểu gắt", "tiểu nhiều lần", "đau hạ vị", "viêm tiết niệu"),
            ("đái rát", "nước tiểu đục", "mệt mỏi", "nhiễm khuẩn tiểu"),
            ("tiểu đêm", "khó chịu bàng quang", "mất ngủ", "rối loạn bài tiết"),
            ("ngứa rát vùng kín", "khó chịu", "dịch bất thường", "viêm nhiễm"),
        ],
    },
    "Kháng sinh & Kháng viêm": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("sốt", "sưng nóng", "đau nhức", "viêm nhiễm"),
            ("đau họng", "sốt nhẹ", "mệt", "nhiễm khuẩn hô hấp"),
            ("viêm lợi", "đau răng", "sưng nướu", "nhiễm khuẩn răng miệng"),
            ("đỏ da", "đau", "nóng vùng tổn thương", "viêm mô mềm"),
        ],
    },
    "Giảm đau & Hạ sốt": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("sốt cao", "đau đầu", "rã rời", "sốt do viêm"),
            ("đau cơ", "ớn lạnh", "nhức mỏi", "cảm cúm"),
            ("đau răng", "đau nửa mặt", "mất ngủ", "đau cấp"),
            ("đau bụng nhẹ", "mệt", "sốt nhẹ", "khó chịu toàn thân"),
        ],
    },
    "Vitamin & Khoáng chất": {
        "patterns": EVAL_BASE_PATTERNS,
        "symptoms": [
            ("uể oải", "ăn kém", "thiếu năng lượng", "thiếu vi chất"),
            ("rụng tóc", "móng giòn", "da xỉn", "thiếu khoáng"),
            ("hoa mắt", "mệt", "giảm tập trung", "thiếu sắt"),
            ("suy nhược", "khó ngủ", "chán ăn", "thiếu vitamin"),
        ],
    },
}

LABELED_QUERIES = build_labeled_queries_from_specs(EVAL_QUERY_SPECS, target_count=240, seed=42)
print(f"Generated LABELED_QUERIES from EVAL_QUERY_SPECS: {len(LABELED_QUERIES)}")
display(pd.DataFrame(LABELED_QUERIES[:10], columns=["query", "expected_category"]))

# fix(2): benchmark ưu tiên tập eval độc lập TEST_QUERIES
if "TEST_QUERIES" in globals() and isinstance(TEST_QUERIES, list) and len(TEST_QUERIES) > 0:
    EVAL_QUERIES = [(q, None) for q in TEST_QUERIES]
    print(f"EVAL_QUERIES built from TEST_QUERIES: {len(EVAL_QUERIES)}")
else:
    EVAL_QUERIES = LABELED_QUERIES
    print("TEST_QUERIES chưa sẵn sàng; fallback tạm sang LABELED_QUERIES")

Generated LABELED_QUERIES from EVAL_QUERY_SPECS: 240


,query,expected_category
0,"Biểu hiện gần đây: nhức khớp gối, sưng nhẹ, đi...",Cơ xương khớp
1,"Xin tư vấn vì có huyết áp cao, thêm mất tập tr...",Tim mạch & huyết áp
2,"Biểu hiện gần đây: hoa mắt, mệt, giảm tập trun...",Vitamin & Khoáng chất
3,"Khoảng 2 hôm nay xuất hiện đái rát, kèm nước t...",Tiết niệu & Sinh dục
4,"Xin tư vấn vì có đau vùng cổ tay, thêm tê nhẹ ...",Cơ xương khớp
5,"Xin tư vấn vì có ngứa rát vùng kín, thêm khó c...",Tiết niệu & Sinh dục
6,"Biểu hiện gần đây: tiểu đêm, khó chịu bàng qua...",Tiết niệu & Sinh dục
7,"Xin tư vấn vì có mụn đỏ, thêm đau rát và tiết ...",Da liễu & dị ứng
8,"Xin tư vấn vì có đau vùng cổ tay, thêm tê nhẹ ...",Cơ xương khớp
9,"Tôi đang bị uể oải, đồng thời ăn kém và thiếu ...",Vitamin & Khoáng chất


EVAL_QUERIES built from TEST_QUERIES: 17


In [33]:
# fix(2) override: EVAL_QUERIES phải có nhãn đầy đủ để metric != 0
EVAL_QUERIES = [
    ("Ho khan 2 ngày, đau họng, sổ mũi, sốt nhẹ 37.8, mệt mỏi.", "Hô hấp"),
    ("Nghẹt mũi, chảy nước mũi, hắt hơi nhiều, ngứa mũi theo mùa.", "Da liễu & dị ứng"),
    ("Ho có đờm đặc, khò khè nhẹ, tức ngực, khó thở khi gắng sức.", "Hô hấp"),
    ("Đau thượng vị, ợ chua, nóng rát sau xương ức, triệu chứng tăng sau ăn.", "Tiêu hóa & gan mật"),
    ("Tiêu chảy nước 5 lần/ngày, đau quặn bụng, không sốt, mất nước nhẹ.", "Tiêu hóa & gan mật"),
    ("Buồn nôn, nôn 2 lần, đầy bụng khó tiêu sau khi ăn đồ lạ.", "Tiêu hóa & gan mật"),
    ("Nổi mề đay, ngứa toàn thân, sẩn đỏ, xuất hiện sau khi ăn hải sản.", "Da liễu & dị ứng"),
    ("Viêm da tiếp xúc: đỏ rát, ngứa vùng cổ tay sau khi đeo đồng hồ.", "Da liễu & dị ứng"),
    ("Mụn viêm mặt, sưng đỏ, nhiều nhân viêm, da dầu.", "Da liễu & dị ứng"),
    ("Đau lưng cấp, co cứng cơ, đau tăng khi vận động, không tê chân.", "Cơ xương khớp"),
    ("Đau khớp gối khi đi lại, sưng nhẹ, hạn chế vận động, đau âm ỉ.", "Cơ xương khớp"),
    ("Đau đầu căng thẳng, đau vùng trán, mất ngủ, stress kéo dài.", "Thần kinh & Não bộ"),
    ("Đỏ mắt, cộm, chảy nước mắt, ngứa mắt nhiều, nghi viêm kết mạc dị ứng.", "Tai, mắt, mũi, họng"),
    ("Viêm họng: đau rát họng, nuốt đau, ho nhẹ, khàn tiếng.", "Hô hấp"),
    ("Sốt cao 39, đau người, đau họng, ho, mệt rã rời.", "Hô hấp"),
    ("Đau bụng, buồn nôn, tiêu chảy, sốt nhẹ.", "Tiêu hóa & gan mật"),
    ("Ngứa, nổi ban đỏ, kèm sổ mũi và hắt hơi.", "Da liễu & dị ứng"),
]
print(f"EVAL_QUERIES labeled size: {len(EVAL_QUERIES)}")
display(pd.DataFrame(EVAL_QUERIES, columns=["query", "expected_category"]))

EVAL_QUERIES labeled size: 17


,query,expected_category
0,"Ho khan 2 ngày, đau họng, sổ mũi, sốt nhẹ 37.8...",Hô hấp
1,"Nghẹt mũi, chảy nước mũi, hắt hơi nhiều, ngứa ...",Da liễu & dị ứng
2,"Ho có đờm đặc, khò khè nhẹ, tức ngực, khó thở ...",Hô hấp
3,"Đau thượng vị, ợ chua, nóng rát sau xương ức, ...",Tiêu hóa & gan mật
4,"Tiêu chảy nước 5 lần/ngày, đau quặn bụng, khôn...",Tiêu hóa & gan mật
5,"Buồn nôn, nôn 2 lần, đầy bụng khó tiêu sau khi...",Tiêu hóa & gan mật
6,"Nổi mề đay, ngứa toàn thân, sẩn đỏ, xuất hiện ...",Da liễu & dị ứng
7,"Viêm da tiếp xúc: đỏ rát, ngứa vùng cổ tay sau...",Da liễu & dị ứng
8,"Mụn viêm mặt, sưng đỏ, nhiều nhân viêm, da dầu.",Da liễu & dị ứng
9,"Đau lưng cấp, co cứng cơ, đau tăng khi vận độn...",Cơ xương khớp


In [34]:
# TF-IDF, TF-IDF + SVD, BiLSTM, BiGRU, TextCNN, PhoBERT
RUN_DL_BENCHMARK = True

if RUN_DL_BENCHMARK:
    import re
    from collections import Counter
    from functools import partial
    from sklearn.neighbors import NearestNeighbors
    from sklearn.metrics.pairwise import cosine_similarity
    from torch import nn
    from torch.utils.data import DataLoader

    DL_BENCHMARK_EPOCHS = BENCHMARK_PROTOCOL["DL_BENCHMARK_EPOCHS"]
    DL_BENCHMARK_BATCH_SIZE = BENCHMARK_PROTOCOL["DL_BENCHMARK_BATCH_SIZE"]
    DL_TEXT_MAX_LENGTH = BENCHMARK_PROTOCOL["DL_TEXT_MAX_LENGTH"]
    DL_MIN_WORD_FREQ = BENCHMARK_PROTOCOL["DL_MIN_WORD_FREQ"]
    DL_MAX_VOCAB_SIZE = BENCHMARK_PROTOCOL["DL_MAX_VOCAB_SIZE"]
    DL_EMBED_DIM = BENCHMARK_PROTOCOL["DL_EMBED_DIM"]
    DL_HIDDEN_DIM = BENCHMARK_PROTOCOL["DL_HIDDEN_DIM"]
    DL_CNN_CHANNELS = BENCHMARK_PROTOCOL["DL_CNN_CHANNELS"]
    DL_PROJ_DIM = BENCHMARK_PROTOCOL["DL_PROJ_DIM"]
    DL_LR = BENCHMARK_PROTOCOL["DL_LR"]
    DL_WEIGHT_DECAY = BENCHMARK_PROTOCOL["DL_WEIGHT_DECAY"]
    DL_TEMPERATURE = BENCHMARK_PROTOCOL["DL_TEMPERATURE"]
    DL_DROPOUT = BENCHMARK_PROTOCOL["DL_DROPOUT"]

    # Benchmark quick-eval ưu tiên EVAL_QUERIES, fallback TEST_QUERIES, cuối cùng LABELED_QUERIES
    if "EVAL_QUERIES" in globals() and isinstance(EVAL_QUERIES, list) and len(EVAL_QUERIES) > 0:
        EVAL_INPUT = EVAL_QUERIES
    elif "TEST_QUERIES" in globals() and isinstance(TEST_QUERIES, list) and len(TEST_QUERIES) > 0:
        EVAL_INPUT = [(q, None) for q in TEST_QUERIES]
    elif "LABELED_QUERIES" in globals() and isinstance(LABELED_QUERIES, list) and len(LABELED_QUERIES) > 0:
        EVAL_INPUT = LABELED_QUERIES
    else:
        EVAL_INPUT = []

    print(f"Benchmark eval set size: {len(EVAL_INPUT)}")

    MODEL_DESCRIPTIONS = {
        "TF-IDF+SVD": "Lexical + SVD (LSA)",
        "BiLSTM": "RNN 2 chiều, train từ đầu",
        "BiGRU": "RNN 2 chiều, train từ đầu",
        "TextCNN": "CNN đa kernel, train từ đầu",
        "PhoBERT": "Transformer pretrained tiếng Việt",
    }

    def simple_word_tokenize(text):
        normalized = normalize_for_match(text)
        normalized = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]+", " ", normalized)
        return [token for token in normalized.split() if token]

    class WordVocab:
        def __init__(self, min_freq=2, max_size=30000):
            self.min_freq = int(min_freq)
            self.max_size = int(max_size)
            self.pad_token = "<pad>"
            self.unk_token = "<unk>"
            self.cls_token = "<cls>"
            self.sep_token = "<sep>"
            self.itos = [self.pad_token, self.unk_token, self.cls_token, self.sep_token]
            self.stoi = {token: idx for idx, token in enumerate(self.itos)}

        @property
        def pad_idx(self):
            return self.stoi[self.pad_token]

        @property
        def unk_idx(self):
            return self.stoi[self.unk_token]

        @property
        def cls_idx(self):
            return self.stoi[self.cls_token]

        @property
        def sep_idx(self):
            return self.stoi[self.sep_token]

        def build(self, texts):
            counter = Counter()
            for text in texts:
                counter.update(simple_word_tokenize(text))
            words = [word for word, freq in counter.items() if freq >= self.min_freq]
            words.sort(key=lambda word: (-counter[word], word))
            if self.max_size:
                words = words[: max(0, self.max_size - len(self.itos))]
            self.itos = [self.pad_token, self.unk_token, self.cls_token, self.sep_token] + words
            self.stoi = {token: idx for idx, token in enumerate(self.itos)}
            return self

        def encode(self, text, max_length=48):
            tokens = [self.cls_token] + simple_word_tokenize(text)[: max(1, max_length - 2)] + [self.sep_token]
            ids = [self.stoi.get(token, self.unk_idx) for token in tokens]
            mask = [1] * len(ids)
            if len(ids) < max_length:
                pad_len = max_length - len(ids)
                ids = ids + [self.pad_idx] * pad_len
                mask = mask + [0] * pad_len
            else:
                ids = ids[:max_length]
                mask = mask[:max_length]
            return ids, mask

        def batch_encode(self, texts, max_length=48):
            encoded = [self.encode(text, max_length=max_length) for text in texts]
            input_ids = torch.tensor([item[0] for item in encoded], dtype=torch.long)
            attention_mask = torch.tensor([item[1] for item in encoded], dtype=torch.long)
            return input_ids, attention_mask

    def collate_word_pair_batch(batch, vocab, max_length):
        query_texts = [item["query_text"] for item in batch]
        doc_texts = [item["doc_text"] for item in batch]
        query_input_ids, query_attention_mask = vocab.batch_encode(query_texts, max_length=max_length)
        doc_input_ids, doc_attention_mask = vocab.batch_encode(doc_texts, max_length=max_length)
        return {
            "query_input_ids": query_input_ids,
            "query_attention_mask": query_attention_mask,
            "doc_input_ids": doc_input_ids,
            "doc_attention_mask": doc_attention_mask,
        }

    def _masked_mean(sequence_output, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (sequence_output * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1.0)
        return summed / denom

    class BiLSTMEncoder(nn.Module):
        def __init__(self, vocab_size, pad_idx, emb_dim=128, hidden_dim=96, proj_dim=256, dropout=0.25):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
            self.dropout = nn.Dropout(dropout)
            self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.projection = nn.Linear(hidden_dim * 2, proj_dim)

        def forward(self, input_ids, attention_mask):
            embedded = self.dropout(self.embedding(input_ids))
            outputs, _ = self.lstm(embedded)
            pooled = _masked_mean(outputs, attention_mask)
            projected = self.projection(self.dropout(pooled))
            return torch.nn.functional.normalize(projected, p=2, dim=1)

    class BiGRUEncoder(nn.Module):
        def __init__(self, vocab_size, pad_idx, emb_dim=128, hidden_dim=96, proj_dim=256, dropout=0.25):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
            self.dropout = nn.Dropout(dropout)
            self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.projection = nn.Linear(hidden_dim * 2, proj_dim)

        def forward(self, input_ids, attention_mask):
            embedded = self.dropout(self.embedding(input_ids))
            outputs, _ = self.gru(embedded)
            pooled = _masked_mean(outputs, attention_mask)
            projected = self.projection(self.dropout(pooled))
            return torch.nn.functional.normalize(projected, p=2, dim=1)

    class TextCNNEncoder(nn.Module):
        def __init__(self, vocab_size, pad_idx, emb_dim=128, cnn_channels=96, proj_dim=256, dropout=0.25):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
            self.dropout = nn.Dropout(dropout)
            self.convs = nn.ModuleList([
                nn.Conv1d(emb_dim, cnn_channels, kernel_size=3, padding=1),
                nn.Conv1d(emb_dim, cnn_channels, kernel_size=5, padding=2),
                nn.Conv1d(emb_dim, cnn_channels, kernel_size=7, padding=3),
            ])
            self.projection = nn.Linear(cnn_channels * len(self.convs), proj_dim)

        def forward(self, input_ids, attention_mask):
            embedded = self.dropout(self.embedding(input_ids)).transpose(1, 2)
            features = []
            for conv in self.convs:
                conv_output = torch.relu(conv(embedded))
                pooled = torch.max(conv_output, dim=2).values
                features.append(pooled)
            merged = torch.cat(features, dim=1)
            projected = self.projection(self.dropout(merged))
            return torch.nn.functional.normalize(projected, p=2, dim=1)

    def train_simple_retriever(model, train_loader, val_loader, epochs, lr, device, patience: int = 3, min_delta: float = 1e-4):
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=DL_WEIGHT_DECAY)
        best_val_loss = float("inf")
        best_state = None
        no_improve = 0

        model.train()
        for epoch in range(int(epochs)):
            total_loss = 0.0
            for batch in train_loader:
                query_ids = batch["query_input_ids"].to(device)
                query_mask = batch["query_attention_mask"].to(device)
                doc_ids = batch["doc_input_ids"].to(device)
                doc_mask = batch["doc_attention_mask"].to(device)

                optimizer.zero_grad(set_to_none=True)
                query_emb = model(query_ids, query_mask)
                doc_emb = model(doc_ids, doc_mask)
                loss = multiple_negatives_ranking_loss(query_emb, doc_emb, temperature=DL_TEMPERATURE, symmetric=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += float(loss.item())

            avg_train_loss = total_loss / max(1, len(train_loader))
            val_loss = None
            if val_loader is not None:
                model.eval()
                with torch.no_grad():
                    val_total = 0.0
                    for batch in val_loader:
                        query_ids = batch["query_input_ids"].to(device)
                        query_mask = batch["query_attention_mask"].to(device)
                        doc_ids = batch["doc_input_ids"].to(device)
                        doc_mask = batch["doc_attention_mask"].to(device)
                        query_emb = model(query_ids, query_mask)
                        doc_emb = model(doc_ids, doc_mask)
                        loss = multiple_negatives_ranking_loss(query_emb, doc_emb, temperature=DL_TEMPERATURE, symmetric=True)
                        val_total += float(loss.item())
                val_loss = val_total / max(1, len(val_loader))
                model.train()

            val_loss_str = f"{val_loss:.4f}" if val_loss is not None else "nan"
            print(f"Epoch {epoch + 1}/{epochs} | train_loss={avg_train_loss:.4f} | val_loss={val_loss_str}")

            if val_loss is None:
                continue
            if val_loss < (best_val_loss - float(min_delta)):
                best_val_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
            if no_improve >= int(patience):
                print(f"Early stop at epoch {epoch + 1}")
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    def encode_simple_texts(model, vocab, texts, device, max_length=48, batch_size=32):
        model.eval()
        all_embeddings = []
        with torch.no_grad():
            for start in range(0, len(texts), batch_size):
                batch_texts = texts[start : start + batch_size]
                input_ids, attention_mask = vocab.batch_encode(batch_texts, max_length=max_length)
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                embeddings = model(input_ids, attention_mask).detach().cpu().numpy().astype(np.float32)
                all_embeddings.append(embeddings)
        return np.vstack(all_embeddings)

    def build_simple_recommender(model, vocab, train_df, device, max_length=48, k_index=10):
        doc_embeddings = encode_simple_texts(
            model,
            vocab,
            train_df[DOCUMENT_TEXT_COL].fillna("").astype(str).tolist(),
            device,
            max_length=max_length,
            batch_size=DL_BENCHMARK_BATCH_SIZE,
        )
        knn_local = NearestNeighbors(n_neighbors=k_index, metric="cosine", algorithm="brute")
        knn_local.fit(doc_embeddings)

        def recommender(query, k=5):
            query_embeddings = encode_simple_texts(model, vocab, [query], device, max_length=max_length, batch_size=1)
            distances, indices = knn_local.kneighbors(query_embeddings, n_neighbors=k)
            scores = 1.0 - distances[0]
            results = train_df.iloc[indices[0]].copy()
            results["score"] = scores
            return results.sort_values("score", ascending=False)

        return recommender

    def recommend_from_query_tfidf_svd(query: str, k: int = 5):
        q_vec = vectorizer.transform([query.strip()])
        q_svd = svd.transform(q_vec).astype(np.float32)
        q_svd = normalize(q_svd)
        dist, idx = knn_tfidf_query.kneighbors(q_svd, n_neighbors=k)
        sims = 1.0 - dist[0]
        results = train_set.iloc[idx[0]].copy()
        results["score"] = sims
        return results.sort_values("score", ascending=False)

    def aggregate_metrics(detail_df: pd.DataFrame, k: int):
        if detail_df is None or len(detail_df) == 0:
            return {"hit@1": np.nan, "hit@3": np.nan, f"hit@{k}": np.nan, f"mrr@{k}": np.nan, f"precision@{k}": np.nan}

        if "expected_category" not in detail_df.columns:
            return {"hit@1": np.nan, "hit@3": np.nan, f"hit@{k}": np.nan, f"mrr@{k}": np.nan, f"precision@{k}": np.nan}

        labeled_df = detail_df[detail_df["expected_category"].notna()].copy()
        if len(labeled_df) == 0:
            return {"hit@1": np.nan, "hit@3": np.nan, f"hit@{k}": np.nan, f"mrr@{k}": np.nan, f"precision@{k}": np.nan}

        return {
            "hit@1": float(labeled_df["hit@1"].mean()),
            "hit@3": float(labeled_df["hit@3"].mean()),
            f"hit@{k}": float(labeled_df[f"hit@{k}"].mean()),
            f"mrr@{k}": float(labeled_df[f"mrr@{k}"].mean()),
            f"precision@{k}": float(labeled_df[f"precision@{k}"].mean()),
        }

    # lưu recommender để cell test_set dùng lại
    dl_recommenders = {}

    word_vocab_texts = list(train_set[DOCUMENT_TEXT_COL].fillna("").astype(str)) + list(train_set[QUERY_TEXT_COL].fillna("").astype(str)) + list(val_set[QUERY_TEXT_COL].fillna("").astype(str))
    word_vocab = WordVocab(min_freq=DL_MIN_WORD_FREQ, max_size=DL_MAX_VOCAB_SIZE).build(word_vocab_texts)
    print("Word vocab size:", len(word_vocab.itos))

    simple_train_dataset = QueryDocumentPairDataset(
        train_set,
        query_col=PHOBERT_QUERY_COL,
        doc_col=PHOBERT_DOC_COL,
        label_col=LABEL_COL,
        pairs_per_row=PAIRS_PER_ROW,
        seed=42,
    )
    simple_val_dataset = QueryDocumentPairDataset(
        val_set,
        query_col=PHOBERT_QUERY_COL,
        doc_col=PHOBERT_DOC_COL,
        label_col=LABEL_COL,
        pairs_per_row=1,
        seed=42,
    )

    train_loader = DataLoader(
        simple_train_dataset,
        batch_size=DL_BENCHMARK_BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        collate_fn=partial(collate_word_pair_batch, vocab=word_vocab, max_length=DL_TEXT_MAX_LENGTH),
    )
    val_loader = DataLoader(
        simple_val_dataset,
        batch_size=DL_BENCHMARK_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        collate_fn=partial(collate_word_pair_batch, vocab=word_vocab, max_length=DL_TEXT_MAX_LENGTH),
    )

    SIMPLE_MODEL_SPECS = [
        ("BiLSTM", BiLSTMEncoder, {"emb_dim": DL_EMBED_DIM, "hidden_dim": DL_HIDDEN_DIM, "proj_dim": DL_PROJ_DIM, "dropout": DL_DROPOUT}),
        ("BiGRU", BiGRUEncoder, {"emb_dim": DL_EMBED_DIM, "hidden_dim": DL_HIDDEN_DIM, "proj_dim": DL_PROJ_DIM, "dropout": DL_DROPOUT}),
        ("TextCNN", TextCNNEncoder, {"emb_dim": DL_EMBED_DIM, "cnn_channels": DL_CNN_CHANNELS, "proj_dim": DL_PROJ_DIM, "dropout": DL_DROPOUT}),
    ]

    benchmark_rows = []
    benchmark_details = {}

    for model_name, model_cls, model_kwargs in SIMPLE_MODEL_SPECS:
        print(f"\n=== Training {model_name} ===")
        model = model_cls(vocab_size=len(word_vocab.itos), pad_idx=word_vocab.pad_idx, **model_kwargs).to(device)
        model = train_simple_retriever(model, train_loader=train_loader, val_loader=val_loader, epochs=DL_BENCHMARK_EPOCHS, lr=DL_LR, device=device, patience=3)

        recommender = build_simple_recommender(model, word_vocab, train_set, device, max_length=DL_TEXT_MAX_LENGTH, k_index=10)
        dl_recommenders[model_name] = recommender

        detail_df = evaluate_labeled_queries_generic(
            EVAL_INPUT,
            k=K_EVAL,
            recommender=recommender,
            method_name=model_name,
        )
        benchmark_details[model_name] = detail_df
        metrics = aggregate_metrics(detail_df, K_EVAL)
        benchmark_rows.append({
            "model": model_name,
            "description": MODEL_DESCRIPTIONS[model_name],
            "n_eval": int(len(detail_df[detail_df["expected_category"].notna()])) if "expected_category" in detail_df.columns else len(EVAL_INPUT),
            "hit@1": metrics["hit@1"],
            "hit@3": metrics["hit@3"],
            f"hit@{K_EVAL}": metrics[f"hit@{K_EVAL}"],
            f"mrr@{K_EVAL}": metrics[f"mrr@{K_EVAL}"],
            f"precision@{K_EVAL}": metrics[f"precision@{K_EVAL}"],
        })

    # TF-IDF+SVD quick row trên cùng bộ EVAL_INPUT để không lệch nguồn metric
    tfidf_detail_df = evaluate_labeled_queries_generic(
        EVAL_INPUT,
        k=K_EVAL,
        recommender=recommend_from_query_tfidf,
        method_name="TF-IDF+SVD",
    )
    benchmark_rows.append({
        "model": "TF-IDF+SVD",
        "description": MODEL_DESCRIPTIONS["TF-IDF+SVD"],
        "n_eval": int(len(tfidf_detail_df[tfidf_detail_df["expected_category"].notna()])) if "expected_category" in tfidf_detail_df.columns else len(EVAL_INPUT),
        **aggregate_metrics(tfidf_detail_df, K_EVAL),
    })

    phobert_recommender = lambda q, k=K_EVAL: recommend_from_query(
        q,
        knn=knn,
        train_emb=train_emb,
        train_df=train_set,
        model=model_attn,
        tokenizer=tokenizer,
        device=device,
        k=k,
        index_max_length=globals().get("INDEX_MAX_LENGTH", None),
    )

    phobert_detail_df = evaluate_labeled_queries_generic(
        EVAL_INPUT,
        k=K_EVAL,
        recommender=phobert_recommender,
        method_name="PhoBERT",
    )
    benchmark_details["PhoBERT"] = phobert_detail_df
    benchmark_rows.append({
        "model": "PhoBERT",
        "description": MODEL_DESCRIPTIONS["PhoBERT"],
        "n_eval": int(len(phobert_detail_df[phobert_detail_df["expected_category"].notna()])) if "expected_category" in phobert_detail_df.columns else len(EVAL_INPUT),
        **aggregate_metrics(phobert_detail_df, K_EVAL),
    })

    bilstm_recommender = dl_recommenders.get("BiLSTM")
    bigru_recommender = dl_recommenders.get("BiGRU")
    textcnn_recommender = dl_recommenders.get("TextCNN")

    comparison_df = pd.DataFrame(benchmark_rows).sort_values(by=f"mrr@{K_EVAL}", ascending=False, na_position="last")
    print("\n=== Benchmark summary (consistent eval input) ===")
    display(comparison_df[["model", "description", "n_eval", "hit@1", "hit@3", f"hit@{K_EVAL}", f"mrr@{K_EVAL}", f"precision@{K_EVAL}"]])
else:
    print("TF-IDF+SVD, BiLSTM, BiGRU, TextCNN, PhoBERT")

Benchmark eval set size: 17
Word vocab size: 13060

=== Training BiLSTM ===
Epoch 1/12 | train_loss=3.3086 | val_loss=2.8264
Epoch 2/12 | train_loss=2.7513 | val_loss=2.6457
Epoch 3/12 | train_loss=2.5771 | val_loss=2.5858
Epoch 4/12 | train_loss=2.4816 | val_loss=2.5265
Epoch 5/12 | train_loss=2.4073 | val_loss=2.5307
Epoch 6/12 | train_loss=2.3611 | val_loss=2.5079
Epoch 7/12 | train_loss=2.3198 | val_loss=2.5093
Epoch 8/12 | train_loss=2.2781 | val_loss=2.5327
Epoch 9/12 | train_loss=2.2467 | val_loss=2.5183
Early stop at epoch 9
[BiLSTM] hit@1=0.824 | hit@3=0.882 | hit@5=0.882 | MRR@5=0.843 | Precision@5=0.800

=== Training BiGRU ===
Epoch 1/12 | train_loss=3.3930 | val_loss=2.8804
Epoch 2/12 | train_loss=2.8078 | val_loss=2.6695
Epoch 3/12 | train_loss=2.6210 | val_loss=2.6090
Epoch 4/12 | train_loss=2.5156 | val_loss=2.5905
Epoch 5/12 | train_loss=2.4493 | val_loss=2.5259
Epoch 6/12 | train_loss=2.3890 | val_loss=2.5450
Epoch 7/12 | train_loss=2.3473 | val_loss=2.5296
Epoch 8/12 

,model,description,n_eval,hit@1,hit@3,hit@5,mrr@5,precision@5
3,TF-IDF+SVD,Lexical + SVD (LSA),17,0.823529,0.941176,0.941176,0.872549,0.788235
4,PhoBERT,Transformer pretrained tiếng Việt,17,0.823529,0.882353,0.882353,0.852941,0.811765
0,BiLSTM,"RNN 2 chiều, train từ đầu",17,0.823529,0.882353,0.882353,0.843137,0.800000
1,BiGRU,"RNN 2 chiều, train từ đầu",17,0.764706,0.941176,0.941176,0.843137,0.776471
2,TextCNN,"CNN đa kernel, train từ đầu",17,0.764706,0.823529,0.823529,0.794118,0.800000


In [36]:

MODEL_DESCRIPTIONS = {
    "TF-IDF+SVD": "Lexical + SVD (LSA)",
    "BiLSTM": "RNN 2 chiều, train từ đầu",
    "BiGRU": "RNN 2 chiều, train từ đầu",
    "TextCNN": "CNN đa kernel, train từ đầu",
    "PhoBERT": "Transformer pretrained tiếng Việt",
}

# Nếu cell benchmark training đã lưu dict thì dùng; nếu không thì fallback biến rời rạc
if "dl_recommenders" not in globals():
    dl_recommenders = {}
if "bilstm_recommender" in globals() and bilstm_recommender is not None:
    dl_recommenders["BiLSTM"] = bilstm_recommender
if "bigru_recommender" in globals() and bigru_recommender is not None:
    dl_recommenders["BiGRU"] = bigru_recommender
if "textcnn_recommender" in globals() and textcnn_recommender is not None:
    dl_recommenders["TextCNN"] = textcnn_recommender

# Quick eval có nhãn (17 câu) để tham khảo định tính
quick_rows = []
if "EVAL_QUERIES" in globals() and len(EVAL_QUERIES) > 0:
    print("=== Bảng tham khảo nhanh — EVAL_QUERIES (17 câu có nhãn) ===")

    def _summarize_eval(detail_df: pd.DataFrame, model_name: str):
        labeled_df = detail_df[detail_df["expected_category"].notna()].copy() if detail_df is not None and len(detail_df) > 0 else pd.DataFrame()
        return {
            "model": model_name,
            "description": MODEL_DESCRIPTIONS.get(model_name, ""),
            "n_eval": int(len(labeled_df)),
            "hit@1": float(labeled_df["hit@1"].mean()) if len(labeled_df) else np.nan,
            "hit@3": float(labeled_df["hit@3"].mean()) if len(labeled_df) else np.nan,
            f"hit@{K_EVAL}": float(labeled_df[f"hit@{K_EVAL}"].mean()) if len(labeled_df) else np.nan,
            f"mrr@{K_EVAL}": float(labeled_df[f"mrr@{K_EVAL}"].mean()) if len(labeled_df) else np.nan,
            f"precision@{K_EVAL}": float(labeled_df[f"precision@{K_EVAL}"].mean()) if len(labeled_df) else np.nan,
        }

    tfidf_quick_df = evaluate_labeled_queries_generic(
        EVAL_QUERIES,
        k=K_EVAL,
        recommender=recommend_from_query_tfidf_svd,
        method_name="TF-IDF+SVD (quick)",
    )
    quick_rows.append(_summarize_eval(tfidf_quick_df, "TF-IDF+SVD"))

    for model_name, rec_fn in [
        ("BiLSTM", dl_recommenders.get("BiLSTM")),
        ("BiGRU", dl_recommenders.get("BiGRU")),
        ("TextCNN", dl_recommenders.get("TextCNN")),
    ]:
        if rec_fn is None:
            continue
        detail_df = evaluate_labeled_queries_generic(EVAL_QUERIES, k=K_EVAL, recommender=rec_fn, method_name=f"{model_name} (quick)")
        quick_rows.append(_summarize_eval(detail_df, model_name))

    pho_quick_df = evaluate_labeled_queries_generic(
        EVAL_QUERIES,
        k=K_EVAL,
        recommender=lambda qq, k=K_EVAL: recommend_from_query(
            qq,
            knn=knn,
            train_emb=train_emb,
            train_df=train_set,
            model=model_attn,
            tokenizer=tokenizer,
            device=device,
            k=k,
            index_max_length=globals().get("INDEX_MAX_LENGTH", None),
        ),
        method_name="PhoBERT (quick)",
    )
    quick_rows.append(_summarize_eval(pho_quick_df, "PhoBERT"))

    quick_comparison_df = pd.DataFrame(quick_rows).sort_values(f"mrr@{K_EVAL}", ascending=False, na_position="last")
    display(quick_comparison_df[["model", "description", "n_eval", "hit@1", "hit@3", f"hit@{K_EVAL}", f"mrr@{K_EVAL}", f"precision@{K_EVAL}"]])
else:
    print("EVAL_QUERIES chưa sẵn sàng; bỏ qua quick benchmark.")

# Chỉ số chính thức trên test_set
print("\n=== KẾT QUẢ CHÍNH THỨC — test_set ===")
test_q = test_set[PHOBERT_QUERY_COL].fillna("").astype(str).tolist()
test_y = test_set[LABEL_COL].fillna("Unknown").astype(str).tolist()

test_rows = []

# Chỉ giữ một dòng baseline: TF-IDF+SVD
m = evaluate_query_doc_retrieval(test_q, test_y, retrieve_fn=recommend_from_query_tfidf_svd, k=K_EVAL)
test_rows.append({"model": "TF-IDF+SVD", "description": MODEL_DESCRIPTIONS["TF-IDF+SVD"], "n_eval": len(test_q), **m})

# PhoBERT
m = evaluate_query_doc_retrieval(
    test_q,
    test_y,
    retrieve_fn=lambda qq, k=K_EVAL: recommend_from_query(
        qq,
        knn=knn,
        train_emb=train_emb,
        train_df=train_set,
        model=model_attn,
        tokenizer=tokenizer,
        device=device,
        k=k,
        index_max_length=globals().get("INDEX_MAX_LENGTH", None),
    ),
    k=K_EVAL,
)
test_rows.append({"model": "PhoBERT", "description": MODEL_DESCRIPTIONS["PhoBERT"], "n_eval": len(test_q), **m})

# DL models (nếu có recommender đã lưu)
for model_name, rec_fn in [("BiLSTM", dl_recommenders.get("BiLSTM")), ("BiGRU", dl_recommenders.get("BiGRU")), ("TextCNN", dl_recommenders.get("TextCNN"))]:
    if rec_fn is None:
        continue
    m = evaluate_query_doc_retrieval(test_q, test_y, retrieve_fn=rec_fn, k=K_EVAL)
    test_rows.append({"model": model_name, "description": MODEL_DESCRIPTIONS[model_name], "n_eval": len(test_q), **m})

test_comparison_df = pd.DataFrame(test_rows).sort_values(f"mrr@{K_EVAL}", ascending=False)
display(test_comparison_df[["model", "description", "n_eval", "hit@1", "hit@3", f"hit@{K_EVAL}", f"mrr@{K_EVAL}", f"precision@{K_EVAL}"]])

if not dl_recommenders:
    print("Lưu ý: chưa có dl_recommenders cho BiLSTM/BiGRU/TextCNN trong session hiện tại.")
    print("Để có đủ 5 model trong bảng test_set, hãy chạy lại Cell benchmark training rồi chạy lại cell patch này.")

=== Bảng tham khảo nhanh — EVAL_QUERIES (17 câu có nhãn) ===
[TF-IDF+SVD (quick)] hit@1=0.824 | hit@3=0.941 | hit@5=0.941 | MRR@5=0.873 | Precision@5=0.788
[BiLSTM (quick)] hit@1=0.824 | hit@3=0.882 | hit@5=0.882 | MRR@5=0.843 | Precision@5=0.800
[BiGRU (quick)] hit@1=0.765 | hit@3=0.941 | hit@5=0.941 | MRR@5=0.843 | Precision@5=0.776
[TextCNN (quick)] hit@1=0.765 | hit@3=0.824 | hit@5=0.824 | MRR@5=0.794 | Precision@5=0.800
[PhoBERT (quick)] hit@1=0.824 | hit@3=0.882 | hit@5=0.882 | MRR@5=0.853 | Precision@5=0.812


,model,description,n_eval,hit@1,hit@3,hit@5,mrr@5,precision@5
0,TF-IDF+SVD,Lexical + SVD (LSA),17,0.823529,0.941176,0.941176,0.872549,0.788235
4,PhoBERT,Transformer pretrained tiếng Việt,17,0.823529,0.882353,0.882353,0.852941,0.811765
1,BiLSTM,"RNN 2 chiều, train từ đầu",17,0.823529,0.882353,0.882353,0.843137,0.800000
2,BiGRU,"RNN 2 chiều, train từ đầu",17,0.764706,0.941176,0.941176,0.843137,0.776471
3,TextCNN,"CNN đa kernel, train từ đầu",17,0.764706,0.823529,0.823529,0.794118,0.800000



=== KẾT QUẢ CHÍNH THỨC — test_set ===


,model,description,n_eval,hit@1,hit@3,hit@5,mrr@5,precision@5
1,PhoBERT,Transformer pretrained tiếng Việt,2131,0.869076,0.913186,0.939465,0.896606,0.857344
4,TextCNN,"CNN đa kernel, train từ đầu",2131,0.865791,0.929141,0.930080,0.889285,0.866542
3,BiGRU,"RNN 2 chiều, train từ đầu",2131,0.829188,0.928672,0.967152,0.884718,0.845519
2,BiLSTM,"RNN 2 chiều, train từ đầu",2131,0.857344,0.873299,0.893947,0.870249,0.848991
0,TF-IDF+SVD,Lexical + SVD (LSA),2131,0.463632,0.579540,0.605350,0.522853,0.475833


In [ ]:
from collections import Counter

if "test_set" in globals() and "recommend_from_query" in globals() and "knn" in globals():
    test_q = test_set[PHOBERT_QUERY_COL].fillna("").astype(str).tolist()
    test_y = test_set[LABEL_COL].fillna("Unknown").astype(str).tolist()

    ranks = []
    for q, y in zip(test_q[:200], test_y[:200]):
        out = recommend_from_query(
            q,
            knn=knn,
            train_emb=train_emb,
            train_df=train_set,
            model=model_attn,
            tokenizer=tokenizer,
            device=device,
            k=5,
            index_max_length=globals().get("INDEX_MAX_LENGTH", None),
        )
        pred = out[LABEL_COL].astype(str).tolist()
        found_rank = None
        for rank, cat in enumerate(pred, start=1):
            if cat == y:
                found_rank = rank
                break
        ranks.append(found_rank)

    rank_counter = Counter(ranks)
    print("Rank distribution for first 200 test queries:")
    print(rank_counter)
    print("Nếu không có rank 4/5 và chỉ có 1/2/3/None thì hit@3=hit@5 là do phân phối rank, không phải bug.")
else:
    print("Thiếu biến cần thiết để kiểm tra rank distribution.")

Rank distribution for first 200 test queries:
Counter({1: 181, None: 13, 3: 3, 2: 2, 4: 1})
Nếu không có rank 4/5 và chỉ có 1/2/3/None thì hit@3=hit@5 là do phân phối rank, không phải bug.


In [38]:


REQUIRED_FIELDS = ["indication", "active_ingredient", "contraindication"]


def _safe_str_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series([""] * len(df), index=df.index, dtype="object")
    return df[col].fillna("").astype(str).str.strip()


def add_quality_columns(df: pd.DataFrame, document_col: str = "document_text") -> pd.DataFrame:
    out = df.copy()

    out["has_indication"] = (_safe_str_series(out, "indication").str.len() > 0).astype(np.int8)
    out["has_active_ingredient"] = (_safe_str_series(out, "active_ingredient").str.len() > 0).astype(np.int8)
    out["has_contraindication"] = (_safe_str_series(out, "contraindication").str.len() > 0).astype(np.int8)

    if document_col not in out.columns:
        out[document_col] = ""
    out["doc_len"] = _safe_str_series(out, document_col).str.len().astype(np.int32)

    # Trọng số ưu tiên chỉ định + hoạt chất
    out["quality_score"] = (
        0.50 * out["has_indication"]
        + 0.30 * out["has_active_ingredient"]
        + 0.20 * out["has_contraindication"]
    ).astype(np.float32)

    out["missing_fields_count"] = (
        3 - (out["has_indication"] + out["has_active_ingredient"] + out["has_contraindication"])
    ).astype(np.int8)

    return out


def quality_dashboard(df: pd.DataFrame, document_col: str = "document_text", name: str = "dataset") -> pd.DataFrame:
    t = add_quality_columns(df, document_col=document_col)

    dashboard = {
        "dataset": name,
        "n_rows": int(len(t)),
        "missing_indication_%": round(float((1 - t["has_indication"].mean()) * 100), 2),
        "missing_active_ingredient_%": round(float((1 - t["has_active_ingredient"].mean()) * 100), 2),
        "missing_contraindication_%": round(float((1 - t["has_contraindication"].mean()) * 100), 2),
        "missing_>=2_fields_%": round(float((t["missing_fields_count"] >= 2).mean() * 100), 2),
        "doc_len_p50": int(np.quantile(t["doc_len"], 0.50)) if len(t) else 0,
        "doc_len_p90": int(np.quantile(t["doc_len"], 0.90)) if len(t) else 0,
        "doc_len_p99": int(np.quantile(t["doc_len"], 0.99)) if len(t) else 0,
        "quality_score_mean": round(float(t["quality_score"].mean()), 4) if len(t) else np.nan,
    }
    return pd.DataFrame([dashboard])


def apply_quality_rerank(
    results: pd.DataFrame,
    base_score_col: str = "score",
    alpha: float = 0.30,
    min_doc_len: int = 40,
    hard_drop_if_missing_2_fields: bool = False,
) -> pd.DataFrame:
    """
    final_score = base_score * (1 - alpha + alpha * quality_score)
    + phạt nhẹ nếu doc quá ngắn.
    """
    if results is None or len(results) == 0:
        return results

    x = add_quality_columns(results, document_col=globals().get("DOCUMENT_TEXT_COL", "document_text")).copy()

    if hard_drop_if_missing_2_fields:
        x = x[x["missing_fields_count"] < 2].copy()
        if len(x) == 0:
            return results

    x["length_penalty"] = np.where(x["doc_len"] < int(min_doc_len), 0.90, 1.00).astype(np.float32)
    x["score_raw"] = x[base_score_col].astype(float)
    x["score_final"] = x["score_raw"] * (1.0 - float(alpha) + float(alpha) * x["quality_score"]) * x["length_penalty"]

    x = x.sort_values(["score_final", "quality_score", "doc_len"], ascending=[False, False, False]).reset_index(drop=True)
    return x


def recommend_from_query_quality_aware(
    query: str,
    k: int = 5,
    alpha: float = 0.30,
    min_doc_len: int = 40,
    hard_drop_if_missing_2_fields: bool = False,
):
    base = recommend_from_query(
        query,
        knn=knn,
        train_emb=train_emb,
        train_df=train_set,
        model=model_attn,
        tokenizer=tokenizer,
        device=device,
        k=max(k, 20),  # lấy rộng hơn để rerank
        index_max_length=globals().get("INDEX_MAX_LENGTH", None),
    )
    reranked = apply_quality_rerank(
        base,
        base_score_col="score",
        alpha=alpha,
        min_doc_len=min_doc_len,
        hard_drop_if_missing_2_fields=hard_drop_if_missing_2_fields,
    )
    return reranked.head(k)


def topk_low_quality_rate(queries, k: int = 5, recommender=None):
    if recommender is None:
        recommender = lambda q, k=k: recommend_from_query_quality_aware(q, k=k)

    rows = []
    for q in queries:
        out = recommender(q, k=k)
        out_q = add_quality_columns(out, document_col=globals().get("DOCUMENT_TEXT_COL", "document_text"))
        rows.append({
            "top1_missing_>=1": float((out_q.head(1)["missing_fields_count"] >= 1).mean()),
            "top1_missing_>=2": float((out_q.head(1)["missing_fields_count"] >= 2).mean()),
            "topk_missing_>=1": float((out_q["missing_fields_count"] >= 1).mean()),
            "topk_missing_>=2": float((out_q["missing_fields_count"] >= 2).mean()),
            "topk_quality_mean": float(out_q["quality_score"].mean()) if len(out_q) else np.nan,
        })

    agg = pd.DataFrame(rows).mean(numeric_only=True).to_dict()
    return {
        "top1_missing_>=1_%": round(float(agg.get("top1_missing_>=1", np.nan) * 100), 2),
        "top1_missing_>=2_%": round(float(agg.get("top1_missing_>=2", np.nan) * 100), 2),
        "topk_missing_>=1_%": round(float(agg.get("topk_missing_>=1", np.nan) * 100), 2),
        "topk_missing_>=2_%": round(float(agg.get("topk_missing_>=2", np.nan) * 100), 2),
        "topk_quality_mean": round(float(agg.get("topk_quality_mean", np.nan)), 4),
    }

In [39]:
# === Quality-aware retrieval + data quality dashboard (RUN) ===

# --- Quick dashboard ---
if all(name in globals() for name in ["train_set", "val_set", "test_set"]):
    dq_train = quality_dashboard(train_set, document_col=DOCUMENT_TEXT_COL, name="train")
    dq_val = quality_dashboard(val_set, document_col=DOCUMENT_TEXT_COL, name="val")
    dq_test = quality_dashboard(test_set, document_col=DOCUMENT_TEXT_COL, name="test")
    dq_all = pd.concat([dq_train, dq_val, dq_test], ignore_index=True)
    print("=== Data quality dashboard ===")
    display(dq_all)

if "TEST_QUERIES" in globals() and isinstance(TEST_QUERIES, list) and len(TEST_QUERIES) > 0:
    print("=== Retrieval quality dashboard (quality-aware recommender) ===")
    print(topk_low_quality_rate(TEST_QUERIES, k=5))


def evaluate_labeled_or_unlabeled_queries(eval_queries, k, recommender, method_name: str):
    rows = []
    for q, expected_cat in eval_queries:
        out = recommender(q, k=k)
        pred_cats = out[LABEL_COL].astype(str).head(k).tolist()

        rank = None
        if expected_cat is not None:
            for i, c in enumerate(pred_cats, start=1):
                if c == expected_cat:
                    rank = i
                    break

        hit1 = 1.0 if rank == 1 else 0.0
        hit3 = 1.0 if (rank is not None and rank <= 3) else 0.0
        hitk = 1.0 if rank is not None else 0.0
        mrr = (1.0 / rank) if rank is not None else 0.0
        precision_k = float(sum([1 if c == expected_cat else 0 for c in pred_cats])) / float(k) if expected_cat is not None else 0.0

        rows.append({
            "query": q,
            "expected_category": expected_cat,
            "top1_category": pred_cats[0] if pred_cats else None,
            "rank": rank,
            "hit@1": hit1,
            "hit@3": hit3,
            f"hit@{k}": hitk,
            f"mrr@{k}": mrr,
            f"precision@{k}": precision_k,
            "top1_drug": out.iloc[0].get("drug_name", None),
            "top1_score": float(out.iloc[0].get("score", np.nan)),
        })

    res = pd.DataFrame(rows)

    labeled = res[res["expected_category"].notna()].copy() if len(res) else pd.DataFrame()
    if len(labeled) > 0:
        print(
            f"[{method_name}] "
            f"hit@1={labeled['hit@1'].mean():.3f} | "
            f"hit@3={labeled['hit@3'].mean():.3f} | "
            f"hit@{k}={labeled[f'hit@{k}'].mean():.3f} | "
            f"MRR@{k}={labeled[f'mrr@{k}'].mean():.3f} | "
            f"Precision@{k}={labeled[f'precision@{k}'].mean():.3f}"
            f"(n_labeled={len(labeled)})"
        )
    else:
        print(f"[{method_name}] Không có nhãn định lượng; hiển thị kết quả định tính top-1 category.")

    return res


K_EVAL = 5
if "EVAL_QUERIES" in globals() and isinstance(EVAL_QUERIES, list) and len(EVAL_QUERIES) > 0:
    EVAL_INPUT = EVAL_QUERIES
elif "TEST_QUERIES" in globals() and isinstance(TEST_QUERIES, list) and len(TEST_QUERIES) > 0:
    EVAL_INPUT = [(q, None) for q in TEST_QUERIES]
elif "LABELED_QUERIES" in globals() and isinstance(LABELED_QUERIES, list) and len(LABELED_QUERIES) > 0:
    EVAL_INPUT = LABELED_QUERIES
else:
    EVAL_INPUT = []

if len(EVAL_INPUT) > 0:
    tfidf_eval_df = evaluate_labeled_or_unlabeled_queries(
        EVAL_INPUT,
        k=K_EVAL,
        recommender=recommend_from_query_tfidf,
        method_name="TF-IDF(SVD)",
    )

    if "knn" in globals() and "train_emb" in globals():
        phobert_eval_df = evaluate_labeled_or_unlabeled_queries(
            EVAL_INPUT,
            k=K_EVAL,
            recommender=lambda q, k=K_EVAL: recommend_from_query(
                q,
                knn=knn,
                train_emb=train_emb,
                train_df=train_set,
                model=model_attn,
                tokenizer=tokenizer,
                device=device,
                k=k,
                index_max_length=globals().get("INDEX_MAX_LENGTH", None),
            ),
            method_name="PhoBERT(index)",
        )

        # Nếu eval không có nhãn -> in thêm phân phối top1 category để so định tính
        if isinstance(EVAL_INPUT[0], (list, tuple)) and len(EVAL_INPUT[0]) >= 2 and EVAL_INPUT[0][1] is None:
            print("\nTop-1 category distribution (TF-IDF):")
            display(tfidf_eval_df["top1_category"].value_counts().head(10))
            print("\nTop-1 category distribution (PhoBERT):")
            display(phobert_eval_df["top1_category"].value_counts().head(10))
else:
    print("Không có tập eval để đánh giá.")

=== Data quality dashboard ===


,dataset,n_rows,missing_indication_%,missing_active_ingredient_%,missing_contraindication_%,missing_>=2_fields_%,doc_len_p50,doc_len_p90,doc_len_p99,quality_score_mean
0,train,18210,4.42,29.97,21.44,9.45,389,468,564,0.8451
1,val,1067,4.87,30.18,22.21,10.12,387,473,576,0.8407
2,test,2131,3.99,28.67,21.07,9.43,390,470,573,0.8519


=== Retrieval quality dashboard (quality-aware recommender) ===
{'top1_missing_>=1_%': 0.0, 'top1_missing_>=2_%': 0.0, 'topk_missing_>=1_%': 4.71, 'topk_missing_>=2_%': 0.0, 'topk_quality_mean': 0.9906}
[TF-IDF(SVD)] hit@1=0.824 | hit@3=0.941 | hit@5=0.941 | MRR@5=0.873 | Precision@5=0.788(n_labeled=17)
[PhoBERT(index)] hit@1=0.824 | hit@3=0.882 | hit@5=0.882 | MRR@5=0.853 | Precision@5=0.812(n_labeled=17)


In [40]:
# Đo độ overlap từ vựng giữa query và document
from sklearn.feature_extraction.text import CountVectorizer

sample = train_set.sample(200, random_state=42)
queries = sample[QUERY_TEXT_COL].fillna("").tolist()
docs = sample[DOCUMENT_TEXT_COL].fillna("").tolist()

overlaps = []
for q, d in zip(queries, docs):
    q_words = set(q.lower().split())
    d_words = set(d.lower().split())
    if len(q_words) > 0:
        overlaps.append(len(q_words & d_words) / len(q_words))

print(f"Query–Document word overlap: mean={np.mean(overlaps):.2%}, p50={np.quantile(overlaps, 0.5):.2%}, p90={np.quantile(overlaps, 0.9):.2%}")

Query–Document word overlap: mean=11.06%, p50=9.09%, p90=27.40%
